# OVO Short-Term Memory Eval — ASI 短期記憶測試

## Cell 1：參數設定

| 參數 | 說明 | 預設值 |
|------|------|--------|
| `ITEM_ID` | 題目 ID（只改這裡） | 483 |
| `BIG_WINDOW_BASE_SIGMA` | 大窗口 SIGMA 公式的 base_sigma | 100.0 |
| `BIG_WINDOW_K` | 大窗口 SIGMA 公式的 k | 1.0 |
| `BIG_WINDOW_EPS` | 大窗口 SIGMA 公式的 ε | 0.01 |
| `BIG_WINDOW_BASE_N` | 大窗口基準取幀數 | 60 |
| `MIN_INTERVAL_BIG` | 大窗口最小間距（秒） | 0.5 |
| `DEDUP_THRESH_D` | 大窗口去重閾值（Cell 6c，Optuna可優化） | 0.95 |
| `DEDUP_THRESH_A` | 突變點展開幀去重閾值（Cell 6d，Optuna可優化） | 0.95 |
| `MAX_N_SMALL` | 突變點每個窗口最多取幾幀 | 10 |
| `MIN_INTERVAL_SMALL` | 突變點窗口最小間距（秒） | 0.25 |
| `K_SIGMOID` | 突變點窗口 sigmoid 下降速率 | 6.0 |
| `SEMANTIC_NEAR_N` | 語意窗口T_event前後各幾幀 | 2 |
| `SEMANTIC_NEAR_INTERVAL` | 語意窗口附近幀間距（秒） | 0.25 |
| `SEMANTIC_FAR_N` | 語意窗口往前/往後各幾幀 | 30 |
| `SEMANTIC_FAR_INTERVAL` | 語意窗口往前/往後幀間距（秒） | 2.0 |
| `REALTIME_NEAR_N` | 即時窗口realtime前後各幾幀 | 2 |
| `REALTIME_NEAR_INTERVAL` | 即時窗口附近幀間距（秒） | 0.25 |
| `REALTIME_FAR_N` | 即時窗口往前幾幀 | 30 |
| `REALTIME_FAR_INTERVAL` | 即時窗口往前幀間距（秒） | 2.0 |
| `BIG_OUTPUT_SIZE` | 大窗口縮圖尺寸（16:9） | (160, 90) |
| `KF_OUTPUT_SIZE` | 突變點展開幀縮圖尺寸（16:9） | (224, 126) |
| `KF_ANCHOR_SIZE` | kf_anchor縮圖尺寸（16:9，1/4原圖） | (640, 360) |
| `STU_MOVE_THRESH` | 位移閾值（畫面比例，Optuna可優化） | 0.02 |
| `STU_AREA_THRESH` | 面積變化率閾值（Optuna可優化） | 0.10 |
| `CLIP_DEDUP_THRESH` | 舊版去重閾值（保留相容） | 0.95 |
| `K` | V4 TDC threshold 係數 | 0.3 |
| `ALPHA` | V4 TDC r₀ 係數 | 1.0 |
| `BETA` | V4 TDC λ 係數 | 1.0 |


In [ ]:
# ==========================================
# Cell 1：參數設定與動態路由 Config (Dynamic Routing)
# ==========================================
ITEM_ID   = 483        # ← 填入測試題號
EXP_NAME  = "v8_valid"

# Google Drive 路徑
DRIVE_ROOT     = '/drive/MyDrive/OVO-Bench'
VIDEO_DIR      = f'{DRIVE_ROOT}/data/chunked_videos/chunked_videos'
ALL_QA_JSON    = f'{DRIVE_ROOT}/data/ovo_bench_new.json'
VIDEO_MAP      = f'{DRIVE_ROOT}/video_path_map.json'

# ── 🏆 Optuna 全題型黃金參數字典 ──
OPTUNA_CONFIGS = {
    'action': {
        'REALTIME_NEAR_N': 6,
        'BIG_WINDOW_BASE_SIGMA': 125.4427,
        'DEDUP_THRESH_D': 0.9900,
        'STU_MOVE_THRESH': 0.0345,
        'K_SIGMOID': 2.1799,
        'SEMANTIC_NEAR_N': 2,
        'DEDUP_THRESH_A': 0.9899
    },
    'attribute': {
        'REALTIME_NEAR_N': 6,
        'BIG_WINDOW_BASE_SIGMA': 146.3926,
        'DEDUP_THRESH_D': 0.9900,
        'STU_MOVE_THRESH': 0.0246,
        'K_SIGMOID': 2.0156,
        'SEMANTIC_NEAR_N': 3,
        'DEDUP_THRESH_A': 0.9899
    },
    'spatial': {
        'REALTIME_NEAR_N': 7,
        'BIG_WINDOW_BASE_SIGMA': 124.8741,
        'DEDUP_THRESH_D': 0.8501,
        'STU_MOVE_THRESH': 0.0115,
        'K_SIGMOID': 3.0265,
        'SEMANTIC_NEAR_N': 5,
        'DEDUP_THRESH_A': 0.9899
    }
}

# ── 釘死的 13 個靜態預設參數 ──
BIG_WINDOW_K          = 1.0
BIG_WINDOW_EPS        = 0.01
BIG_WINDOW_BASE_N     = 16
MIN_INTERVAL_BIG      = 0.5
MIN_INTERVAL_SMALL    = 0.25
MAX_N_SMALL      = 8
SEMANTIC_NEAR_INTERVAL = 0.25
REALTIME_NEAR_INTERVAL = 0.25
STU_AREA_THRESH       = 0.10
CLIP_DEDUP_THRESH     = 0.95
K                     = 0.3
ALPHA                 = 1.0
BETA                  = 1.0

FIB_OFFSETS = [1, 2, 3, 5, 8, 13, 21, 34, 55]

# 輸出圖片尺寸
BIG_OUTPUT_SIZE = (240, 135)   # D-Window
KF_OUTPUT_SIZE  = (240, 135)   # A-Window
KF_ANCHOR_SIZE  = (800, 450)   # 參考用，實際改為動態長邊計算

ANTHROPIC_API_KEY     = 'YOUR_API_KEY_HERE'  # ← 填入 API key
OUT_DIR               = '/content/ovo_stm_frames'


## Cell 2：環境安裝


In [ ]:
!pip install yt-dlp -q
!pip install git+https://github.com/openai/CLIP.git -q
!apt-get install -y ffmpeg -q
!pip install scipy anthropic ultralytics spacy openai-whisper -q
!python -m spacy download en_core_web_sm -q
!pip install ipynbname -q
print('✅ 環境安裝完成')


## Cell 3：讀題目 + 從 Google Drive 載入影片

從 Drive 的 `ASI_questions.json` 讀取指定 `ITEM_ID` 的題目，
直接使用已下載的切片影片，不需要 YouTube 下載。


In [ ]:
import json, os
from google.colab import drive

drive.mount('/drive')

with open(ALL_QA_JSON) as f:
    all_qa = json.load(f)

with open(VIDEO_MAP) as f:
    video_map = json.load(f)

print(f'✅ 題庫載入完成，共 {len(all_qa)} 題')
print(f'✅ 影片對應表載入完成，共 {len(video_map)} 部影片')

# --- 新增：載入加權分數解答本 ---
import json
import os

weight_json_path = '/drive/MyDrive/OVO-Bench/question_weights.json'

if os.path.exists(weight_json_path):
    with open(weight_json_path, 'r', encoding='utf-8') as f:
        weight_table = json.load(f)
    print(f"📊 成功載入權重解答本，共計 {len(weight_table)} 題。")
else:
    weight_table = {}
    print("⚠️ 找不到權重解答本，將以預設 0 分計算。")
# ----------------------------


## Cell 4：載入模型
CLIP、YOLOv8n、YOLO-World、CLIP-Seg、spaCy 只載入一次，所有題目共用。


In [ ]:
import cv2, torch, clip, numpy as np
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import norm
from ultralytics import YOLO
import spacy

import subprocess
subprocess.run(['pip', 'install', 'transformers', '-q'], capture_output=True)

from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
from transformers import AutoProcessor, AutoModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用裝置：{device}')

# SigLIP 2（全片語意編碼 + 文字對齊 + delta_embeddings）
siglip_model     = AutoModel.from_pretrained('google/siglip2-base-patch16-224').to(device)
siglip_processor = AutoProcessor.from_pretrained('google/siglip2-base-patch16-224')
siglip_model.eval()
print('✅ SigLIP 2 載入完成')

# YOLOv8n（保留用於 E 來源）
yolo_model = YOLO('yolov8n.pt')

# YOLO-World（open-vocabulary，用於 Cell 7c）
yolo_world = YOLO('yolov8s-worldv2.pt')

# CLIP-Seg（語意分割，用於 Cell 7d）
clipseg_processor = CLIPSegProcessor.from_pretrained('CIDAS/clipseg-rd64-refined')
clipseg_model     = CLIPSegForImageSegmentation.from_pretrained('CIDAS/clipseg-rd64-refined')
clipseg_model     = clipseg_model.to(device)
clipseg_model.eval()

# spaCy
nlp = spacy.load('en_core_web_sm')

print('✅ SigLIP 2 + YOLOv8n + YOLO-World + CLIP-Seg + spaCy 載入完成')

## Cell 5：CLIP 編碼全片 + V4 TDC
對整支影片逐秒編碼，**只跑一次**，所有題目共用。


In [ ]:
import os

# ==========================================
# 🌟 新增：自動取得目前題目的影片路徑 (RAW_PATH)
# ==========================================
# 1. 從題庫 (all_qa) 找出這題的資料
item_data = next((item for item in all_qa if item['id'] == ITEM_ID), None)
if not item_data:
    raise ValueError(f"❌ 題庫中找不到 ITEM_ID={ITEM_ID} 的題目！")

# 2. 取得影片的 ID 或名稱 (依照你的 JSON 結構，通常是 youtube_id 或 video)
vid_id = item_data.get('youtube_id', item_data.get('video_name', str(ITEM_ID)))

# 3. 透過 video_map 找出實體檔案名稱，並加上 VIDEO_DIR
if vid_id in video_map:
    # 假設 video_map 存的是相對路徑或檔名
    RAW_PATH = os.path.join(VIDEO_DIR, video_map[vid_id])
else:
    # 備用方案：直接假設是 .mp4 結尾
    RAW_PATH = os.path.join(VIDEO_DIR, f"{vid_id}.mp4")

print(f"🎬 準備讀取影片: {RAW_PATH}")
# ==========================================

# ── 以下接續原本的程式碼 ──
cap = cv2.VideoCapture(RAW_PATH)
fps_vid    = cap.get(cv2.CAP_PROP_FPS)
total_sec  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps_vid)
print(f'影片長度：{total_sec}s  fps={fps_vid:.1f}')

# ── 固定步長CLIP取樣 ──
STEP = 1.0  # 固定步長（秒）

embeddings, timestamps = [], []
t = 0.0

while t <= total_sec:
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
    ret, frame = cap.read()
    if not ret:
        t += STEP
        continue
    img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    inputs = siglip_processor(images=img_pil, return_tensors='pt').to(device)
    with torch.no_grad():
        emb = siglip_model.vision_model(**inputs).pooler_output.detach().cpu().numpy()[0]
    emb_norm = emb / (np.linalg.norm(emb) + 1e-8)
    embeddings.append(emb_norm)
    timestamps.append(round(t, 2))
    t = round(t + STEP, 2)

cap.release()
embeddings = np.array(embeddings)
timestamps = np.array(timestamps)
total_sec  = float(timestamps[-1]) if len(timestamps) > 0 else total_sec
print(f'✅ SigLIP 2 固定步長編碼：{len(timestamps)} 幀（步長={STEP}s）')

# S2 探針 + OLS 校正
s2_mask = [i for i, t in enumerate(timestamps) if t <= 20 or (60 <= t <= 100)]
if len(s2_mask) < 2: s2_mask = list(range(len(timestamps)))
s2_embs  = embeddings[s2_mask]
s2_diffs = [1 - cosine_similarity(s2_embs[j:j+1], s2_embs[j+1:j+2])[0][0]
            for j in range(len(s2_embs)-1)]
s2_mu       = float(np.mean(s2_diffs))
s2_sigma    = float(np.std(s2_diffs))
_corr = np.corrcoef(
    s2_embs[:-1].mean(axis=1), s2_embs[1:].mean(axis=1)
)[0,1] if len(s2_embs) > 2 else 0.0
s2_autocorr = 0.0 if np.isnan(_corr) else float(_corr)

full_mu    = 0.8142 * s2_mu    + 0.1809
full_sigma = max(0.8074 * s2_sigma + 0.0087, 0.001)

r0  = full_mu + K * full_sigma
lam = BETA  * (1 - s2_autocorr) / total_sec
print(f'S2: μ={s2_mu:.4f} σ={s2_sigma:.4f} autocorr={s2_autocorr:.4f}')
print(f'V4: r₀={r0:.4f} λ={lam:.4f}')

# V4 TDC 全片關鍵幀
clusters, kf_all = [], []
for emb, t in zip(embeddings, timestamps):
    if not clusters:
        clusters.append((emb, t)); kf_all.append(t); continue
    eff_dists = [
        (1 - cosine_similarity(emb.reshape(1,-1), c.reshape(1,-1))[0][0])
        - r0 * np.exp(-lam * (t - born))
        for c, born in clusters
    ]
    if min(eff_dists) > 0:
        clusters.append((emb, t)); kf_all.append(t)
    else:
        idx_n = int(np.argmin([
            1 - cosine_similarity(emb.reshape(1,-1), c.reshape(1,-1))[0][0]
            for c, _ in clusters
        ]))
        old_c, old_b = clusters[idx_n]
        new_c = (old_c + emb) / 2
        new_c = new_c / np.linalg.norm(new_c)
        clusters[idx_n] = (new_c, old_b)
print(f'✅ V4 TDC：{len(kf_all)} 個全片關鍵幀')

# ── Whisper 逐字稿（只跑一次，全片共用）──
import subprocess, whisper as whisper_lib

# 先檢查是否有音軌
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-select_streams', 'a',
     '-show_entries', 'stream=codec_type', '-of', 'csv=p=0', RAW_PATH],
    capture_output=True, text=True
)

if 'audio' in probe.stdout:
    whisper_model = whisper_lib.load_model('base')
    whisper_result = whisper_model.transcribe(RAW_PATH, language='en')
    whisper_segments = whisper_result['segments']
    print(f'✅ Whisper 完成：{len(whisper_segments)} 段逐字稿')
else:
    whisper_segments = []
    print('⚠️ 無音軌，跳過 Whisper，逐字稿為空')

# ── 差分向量（動詞語意用，供 Cell 6b T_event 定位）──
delta_embeddings = embeddings[1:] - embeddings[:-1]
delta_norms = np.linalg.norm(delta_embeddings, axis=1, keepdims=True)
delta_norms[delta_norms == 0] = 1e-8
delta_embeddings = delta_embeddings / delta_norms  # L2 正規化
print(f'✅ 差分向量：{delta_embeddings.shape}')

# 影片總長度（供 Cell 6c 使用）
T_total = float(total_sec)
print(f'T_total = {T_total:.1f}s')


## Cell 6：選幀總覽

本系統採用多層波形疊加架構，逐題選幀：

1. **Cell 6b**：T_event定位（多詞加權CLIP向量）
2. **Cell 6c**：大窗口（D）動態取樣 + 去重
3. **Cell 6d**：突變點窗口（A）sigmoid逆CDF取樣 + 去重
4. **Cell 6e**：語意窗口（C）固定間距取樣
5. **Cell 6f**：即時窗口（B）固定間距取樣
6. **Cell 6g**：統一時間戳 + 來源標籤整合


## Cell 6b：T_event 定位（多詞加權版）

根據問題類型選擇不同的 T_event 定位策略：

| 問題類型 | 判斷條件 | 策略 |
|---|---|---|
| `attribute` | color / how many / where is / label 等 | 名詞向量加權和 × 全片原始幀 |
| `spatial` | direction / position / relative / 方向詞 | 名詞向量（主）+ 方向詞向量（輔）× 全片原始幀 |
| `action` | 其他（含時序推理）| 名詞×原始幀（0.6）+ 動詞×差分幀（0.4）|

**輸入：** `embeddings`、`timestamps`、`delta_embeddings`（來自 Cell 5）

**輸出：** `item_tevent_map`（dict，key=item_id，供 Cell 6 使用）


In [ ]:
# ── extract_keywords（Cell 6 會用到，這裡先定義供 Cell 6b 使用）──
def extract_keywords(question):
    doc = nlp(question)
    nouns     = [t.text for t in doc if t.pos_ in ('NOUN','PROPN')]
    verbs     = [t.text for t in doc if t.pos_ == 'VERB']
    dirs      = [t.text for t in doc if t.text.lower() in
                  ('left','right','front','behind','above','below','forward','backward')]
    is_after  = any(t.text.lower() == 'after'  for t in doc)
    is_before = any(t.text.lower() == 'before' for t in doc)
    return nouns, verbs, dirs, is_after, is_before

# ═══════════════════════════════════════════════════════════
# Cell 6b：T_event 定位（多詞加權版）
# ═══════════════════════════════════════════════════════════

def classify_question(question, nouns, verbs, dirs):
    """問題類型分類器 V2（加入防呆優先級與擴充詞庫）"""
    q_lower = question.lower()

    # 🌟 絕對優先級 1：屬性細節 (問顏色、數量、特定物品)
    if any(w in q_lower for w in ['color', 'colour', 'how many', 'count', 'what food', 'what item']):
        return 'attribute'

    # 🌟 攔截例外：預測未來的動作 (必須看最後的動態)
    if any(w in q_lower for w in ['going to do', 'about to do', 'planning to do', 'preparing to']):
        return 'action'

    # 🌟 攔截例外：詢問手腳等身體部位 (避免被誤判為空間)
    if 'hand' in q_lower or 'foot' in q_lower or 'finger' in q_lower:
        return 'action'

    # 🌟 優先級 2：空間位置 (擴充了超多方位詞)
    if dirs or any(w in q_lower for w in [
        'where is', 'where was', 'where did', 'where do',
        'left', 'right', 'top', 'bottom', 'under', 'above', 'around', 'behind', 'front', 'direction'
    ]):
        return 'spatial'

    # 預設：動作類 (Action)
    return 'action'


def encode_words(words):
    if not words:
        return []
    embs = []
    for w in words:
        inputs = siglip_processor(text=[w], return_tensors='pt', padding=True).to(device)
        with torch.no_grad():
            e = siglip_model.text_model(**inputs).pooler_output.detach().cpu().numpy()
        norm = np.linalg.norm(e)
        if norm > 0:
            e = e / norm
        else:
            e = np.zeros_like(e)
        embs.append(e)
    return embs


# ── Bug 1 修正：Cell 12（find_t_event 所在）──
# 只需替換 weighted_sim 函數本體，其餘不動

def safe_cosine_similarity(a, b):
    """防呆版 cosine_similarity，過濾 NaN 和全零向量"""
    if np.any(np.isnan(a)) or np.any(np.isnan(b)):
        rows = a.shape[0]
        cols = b.shape[0] if b.ndim > 1 else 1
        return np.zeros((rows, cols))
    if np.all(a == 0) or np.all(b == 0):
        rows = a.shape[0]
        cols = b.shape[0] if b.ndim > 1 else 1
        return np.zeros((rows, cols))
    return cosine_similarity(a, b)


def weighted_sim(emb_list, weights, target_embeddings):
    """多向量加權 cosine similarity，回傳 (N,)"""
    if not emb_list:
        return np.zeros(len(target_embeddings))
    sims    = np.zeros(len(target_embeddings))
    total_w = 0.0
    for emb, w in zip(emb_list, weights):
        # 跳過全零或含 NaN 的向量
        if np.any(np.isnan(emb)) or np.all(emb == 0):
            continue
        s = safe_cosine_similarity(emb, target_embeddings)[0]
        sims    += w * s
        total_w += w
    if total_w > 0:
        sims /= total_w
    return sims


def find_t_event(question, nouns, verbs, dirs,
                 embeddings, timestamps, delta_embeddings):
    """
    多詞加權 T_event 定位
    回傳：T_event（秒）、q_emb（供大窗口邊界用）、q_type、sims
    """
    q_type    = classify_question(question, nouns, verbs, dirs)
    noun_embs = encode_words(nouns)
    verb_embs = encode_words(verbs)
    dir_embs  = encode_words(dirs)

    if q_type == 'attribute':
        # 名詞加權和 → argmax
        sims = weighted_sim(noun_embs, [1.0] * len(noun_embs), embeddings)

    elif q_type == 'spatial':
        # 名詞（主）+ 方向詞（輔）
        noun_sims = weighted_sim(noun_embs, [1.0] * len(noun_embs), embeddings)
        dir_sims  = weighted_sim(dir_embs,  [0.5] * len(dir_embs),  embeddings)
        sims = noun_sims + dir_sims

    else:  # action（含時序推理）
        # 名詞 × 原始幀
        noun_sims = weighted_sim(noun_embs, [1.0] * len(noun_embs), embeddings)
        # 動詞 × 差分幀（語意微分）
        if verb_embs and len(delta_embeddings) > 0:
            verb_sims_delta = weighted_sim(
                verb_embs, [1.0] * len(verb_embs), delta_embeddings)
            verb_sims = np.zeros(len(embeddings))
            verb_sims[:len(verb_sims_delta)] = verb_sims_delta
        else:
            verb_sims = np.zeros(len(embeddings))
        # 合併：名詞 0.6 + 動詞 0.4
        sims = 0.6 * noun_sims + 0.4 * verb_sims

    # T_event
    best_idx = int(np.argmax(sims))
    T_event  = float(timestamps[best_idx])

    # q_emb：名詞+動詞均值向量（供大窗口邊界查詢用）
    all_embs = noun_embs + verb_embs
    if all_embs:
        q_emb = np.mean(np.concatenate(all_embs, axis=0), axis=0, keepdims=True)
    else:
        q_emb = np.zeros((1, embeddings.shape[1]))

    return T_event, q_emb, q_type, sims


# ── 對所有題目預計算 T_event 並「動態切換黃金參數」 ──
item_tevent_map = {}

# (注意：如果你的程式碼是單題測試，迴圈變數可能叫 item_data)
# 這裡以支援單題或多題的清單為例
items_to_process = [i for i in all_qa if i['id'] == ITEM_ID] if 'ITEM_ID' in globals() else all_items

for item in items_to_process:
    item_id  = item['id']
    question = item['question']
    nouns, verbs, dirs, is_after, is_before = extract_keywords(question)

    T_event, q_emb, q_type, sims = find_t_event(
        question, nouns, verbs, dirs,
        embeddings, timestamps, delta_embeddings
    )

    # ==========================================
    # 🌟 動態路由：根據 q_type 載入專屬黃金參數
    # ==========================================
    config = OPTUNA_CONFIGS.get(q_type, OPTUNA_CONFIGS['action']) # 預設 fallback 為 action

    REALTIME_NEAR_N       = config['REALTIME_NEAR_N']
    BIG_WINDOW_BASE_SIGMA = config['BIG_WINDOW_BASE_SIGMA']
    DEDUP_THRESH_D        = config['DEDUP_THRESH_D']
    STU_MOVE_THRESH       = config['STU_MOVE_THRESH']
    K_SIGMOID             = config['K_SIGMOID']
    SEMANTIC_NEAR_N       = config['SEMANTIC_NEAR_N']
    DEDUP_THRESH_A        = config['DEDUP_THRESH_A']

    print(f"\n⚙️ 系統偵測到【{q_type.upper()}】題型，已自動切換對應黃金參數裝備！")
    # ==========================================

    item_tevent_map[item_id] = {
        'T_event':   T_event,
        'q_emb':     q_emb,
        'q_type':    q_type,
        'is_after':  is_after,
        'is_before': is_before,
        'nouns':     nouns,
        'verbs':     verbs,
        'dirs':      dirs,
    }

    print(f'ID={item_id:4d}  [{q_type:9s}]  T_event={T_event:6.1f}s  '
          f'nouns={nouns}  verbs={verbs}  dirs={dirs}')

print(f'\n✅ 共處理 {len(item_tevent_map)} 題')


## Cell 6c：大窗口動態參數計算 + 取樣 + 去重

根據冷啟動層的 `full_mu` 和影片長度 `T_total`，動態計算大窗口參數並完成取樣與去重。

**參數計算：**

| 參數 | 公式 | 防禦條件 |
|---|---|---|
| `BIG_WINDOW_N` | `int(base_n × min(1.0, T/600))` | 最少5幀 |
| `BIG_WINDOW_SIGMA` | `base_sigma × ln(k / (full_mu + ε))` | inner≤0時fallback為base_sigma；最小5秒 |

**初始參數：** `base_sigma=100`、`k=1.0`、`ε=0.01`、`base_n=60`

**取樣方式：**
- 窗口範圍：`[T_event − 2×SIGMA, T_event + 2×SIGMA]`，裁剪至影片邊界
- 逆CDF高斯取樣：以T_event為中心，BIG_WINDOW_N幀散布在整個窗口
- 強制加入T_event本身

**去重：**
- T_event強制保留
- 其他幀：CLIP cosine similarity > `DEDUP_THRESH_D`（0.95）則去掉

**Optuna可優化：** `base_sigma`、`k`、`ε`、`base_n`、`DEDUP_THRESH_D`

**輸入：** `full_mu`、`T_total`（Cell 9）、`item_tevent_map`（Cell 6b）

**輸出：** `item_bigwindow_map`（含去重後的`big_times`、`win_min`、`win_max`）


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 6c：大窗口動態參數計算
# ═══════════════════════════════════════════════════════════
import numpy as np

def compute_big_window_params(full_mu, T_total, q_type, is_after, is_before,
                               base_sigma=BIG_WINDOW_BASE_SIGMA,
                               k=BIG_WINDOW_K,
                               eps=BIG_WINDOW_EPS,
                               base_n=BIG_WINDOW_BASE_N,
                               asym_ratio=0.7):
    """
    動態計算大窗口參數

    BIG_WINDOW_N     = int(base_n × min(1.0, T / 600))，最少5幀
    BIG_WINDOW_SIGMA = base_sigma × ln(k / (full_mu + eps))，最小5秒
                       若 inner<=0 則 fallback 為 base_sigma
    ASYM_RATIO       = 0.5（非時序類）或 asym_ratio（時序類，Optuna優化）
                       main_n 和 other_n 各至少1幀
    """
    # BIG_WINDOW_N：最少5幀
    big_n = max(int(base_n * min(1.0, T_total / 600)), 5)

    # BIG_WINDOW_SIGMA
    denom = full_mu + eps
    inner = k / denom
    if inner <= 0:
        big_sigma = base_sigma
    else:
        big_sigma = max(round(base_sigma * np.log(inner), 1), 5.0)

    # ASYM_RATIO
    if q_type == 'temporal':
        ratio = asym_ratio
    else:
        ratio = 0.5

    # main_n 和 other_n 各至少1幀
    main_n  = max(1, int(big_n * ratio))
    other_n = max(1, big_n - main_n)

    return {
        'BIG_WINDOW_N':     big_n,
        'BIG_WINDOW_SIGMA': big_sigma,
        'MAIN_SIDE_N':      main_n,
        'OTHER_SIDE_N':     other_n,
        'ASYM_RATIO':       ratio,
    }


# ── 對所有題目計算大窗口參數 ──
item_bigwindow_map = {}

for item_id, meta in item_tevent_map.items():
    params = compute_big_window_params(
        full_mu   = full_mu,
        T_total   = T_total,
        q_type    = meta['q_type'],
        is_after  = meta['is_after'],
        is_before = meta['is_before'],
    )
    # 大窗口範圍
    T_event = item_tevent_map[item_id]['T_event']
    sigma   = params['BIG_WINDOW_SIGMA']
    win_min = round(max(0, T_event - 2 * sigma), 1)
    win_max = round(min(T_total, T_event + 2 * sigma), 1)
    params['win_min'] = win_min
    params['win_max'] = win_max

    # 大窗口取樣（逆CDF高斯）
    x = np.linspace(0, 1, 1000)
    duration = win_max - win_min
    if duration > 0:
        center_rel = np.clip((T_event - win_min) / duration, 0, 1)
        sigma_rel  = sigma / duration
        density    = np.exp(-0.5 * ((x - center_rel) / sigma_rel) ** 2)
        cdf        = np.cumsum(density)
        cdf        = cdf / cdf[-1]
        cdf_targets   = np.linspace(0, 1, params['BIG_WINDOW_N'] + 2)[1:-1]
        x_positions   = np.interp(cdf_targets, cdf, x)
        big_times = sorted([round(float(win_min + xp * duration), 1) for xp in x_positions])
    else:
        big_times = [round(T_event, 1)]
    # 強制加入T_event
    t_event_r = round(T_event, 1)
    if t_event_r not in big_times:
        big_times.append(t_event_r)
        big_times = sorted(big_times)
    # ── 大窗口去重（CLIP cosine similarity，閾值DEDUP_THRESH_D）──

    T_event_r = round(T_event, 1)
    kept      = []
    kept_embs = []
    for t in big_times:
        t_int = max(0, min(int(round(t)), len(embeddings) - 1))
        emb   = embeddings[t_int:t_int+1]
        if t == T_event_r:
            # T_event 強制保留
            kept.append(t)
            kept_embs.append(emb)
        elif kept_embs:
            vstack = np.vstack(kept_embs)
            # 【修正】NaN 防呆：跳過問題向量，保守保留該幀
            if np.any(np.isnan(emb)) or np.any(np.isnan(vstack)):
                kept.append(t)
                kept_embs.append(emb)
                continue
            sims = cosine_similarity(emb, vstack)[0]
            if sims.max() <= DEDUP_THRESH_D:
                kept.append(t)
                kept_embs.append(emb)
        else:
            kept.append(t)
            kept_embs.append(emb)
    big_times        = kept
    params['big_times'] = big_times

    item_bigwindow_map[item_id] = params
    print(f'ID={item_id:4d}  [{meta["q_type"]:9s}]  '
          f'N={params["BIG_WINDOW_N"]:3d}  SIGMA={params["BIG_WINDOW_SIGMA"]:6.1f}s  '
          f'win=[{win_min:.0f}s~{win_max:.0f}s]  去重後={len(big_times)}幀')
    print(f'  大窗口取樣：{big_times}')

print(f'\n✅ 共處理 {len(item_bigwindow_map)} 題')


## Cell 6d：突變點窗口取樣 + 去重

每道題在大窗口範圍內找V4突變點，建立窗口序列並做sigmoid逆CDF取樣，取樣後立即去重。

**流程：**
1. 在 `win_min ~ win_max` 內找最近5個V4突變點（不含T_event）
2. 將T_event插入序列排序，形成最多6個窗口
3. 每個窗口寬度 = 相鄰點距離，最小5s（可重疊下一窗口），最大60s
4. 每個窗口sigmoid逆CDF取樣，密度由高（靠近起點）到低（靠近終點），每窗口最多10幀
5. 強制加入每個窗口起點（突變點本身）
6. 去重：kf_anchor強制保留，展開幀CLIP cosine similarity > `DEDUP_THRESH_A`（0.95）則去掉

**特殊情況：**
- 無突變點 → 只有T_event，窗口固定60s
- 窗口寬度 < 5s → 強制拉到5s（可重疊下一窗口）
- 窗口寬度 > 60s → 截斷為60s

**Optuna可優化：** `K_SIGMOID`、`MAX_N_SMALL`、`DEDUP_THRESH_A`

**輸入：** `item_tevent_map`、`item_bigwindow_map`、`kf_all`、`total_sec`

**輸出：** `item_kf_windows_map`（去重後，供 Cell 7b 使用）


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 6d：突變點窗口取樣（新設計）
# ═══════════════════════════════════════════════════════════
import numpy as np

def sigmoid_sample(t_start, t_end, k_sigmoid=K_SIGMOID, min_interval=MIN_INTERVAL_SMALL):
    duration = t_end - t_start
    if duration <= 0:
        return [round(t_start, 2)]

    x = np.linspace(0, 1, 1000)
    density = 1 - 1/(1 + np.exp(-k_sigmoid * (x - 0.5)))
    cdf = np.cumsum(density)
    cdf = cdf / cdf[-1]

    # n 由 duration 和 min_interval 自然決定
    n = min(max(1, int(duration / min_interval)), MAX_N_SMALL)

    cdf_targets = np.linspace(0, 1, n+2)[1:-1]
    x_positions = np.interp(cdf_targets, cdf, x)
    times = t_start + x_positions * duration
    return [round(float(t), 2) for t in sorted(times)]


def compute_kf_windows(item_id, item_tevent_map, item_bigwindow_map,
                        kf_all, total_sec):
    """
    建立突變點窗口序列並取樣
    回傳：list of { 'window': (t_start, t_end), 'sampled': [t, ...] }
    """
    meta    = item_tevent_map[item_id]
    bw      = item_bigwindow_map[item_id]
    T_event = meta['T_event']
    sigma   = bw['BIG_WINDOW_SIGMA']
    win_min = max(0, T_event - 2 * sigma)
    win_max = min(total_sec, T_event + 2 * sigma)

    # 1. 找最近3個突變點（不含T_event），依 after/before 配置主次方向
    is_after  = meta['is_after']
    is_before = meta['is_before']

    kf_before = sorted(
        [t for t in kf_all if win_min <= t < T_event],
        key=lambda t: abs(t - T_event)
    )
    kf_after = sorted(
        [t for t in kf_all if T_event < t <= win_max],
        key=lambda t: abs(t - T_event)
    )

    if is_after and not is_before:
        main, other = kf_after, kf_before
    elif is_before and not is_after:
        main, other = kf_before, kf_after
    else:
        main, other = kf_after, kf_before

    main_picks  = main[:2]
    other_picks = other[:1]

    if not main_picks:
        main_picks  = other[:2]
        other_picks = []

    kf_candidates = sorted(main_picks + other_picks)

    # 2. 插入T_event並排序
    kf_sequence = sorted(kf_candidates + [T_event])

    # 3. 建立窗口
    windows = []
    for i, t_start in enumerate(kf_sequence):
        if i + 1 < len(kf_sequence):
            t_end = kf_sequence[i + 1]
        else:
            t_end = min(t_start + 60, win_max, total_sec)
        t_end = max(t_end, t_start + 5)       # 最小5秒
        t_end = min(t_end, t_start + 60,      # 最大60秒
                    win_max, total_sec)
        windows.append((round(t_start, 1), round(t_end, 1)))

    # 4. 每個窗口逆CDF取樣，強制加入窗口起點（突變點本身）
    result = []
    for t_start, t_end in windows:
        sampled = sigmoid_sample(t_start, t_end)
        # 強制加入窗口起點
        t_start_r = round(t_start, 2)
        if t_start_r not in sampled:
            sampled.append(t_start_r)
            sampled = sorted(sampled)
        result.append({'window': (t_start, t_end), 'sampled': sampled})

    return result, kf_sequence


# ── 對所有題目預計算突變點窗口 ──
item_kf_windows_map = {}

for item_id, meta in item_tevent_map.items():
    windows_result, kf_seq = compute_kf_windows(
        item_id, item_tevent_map, item_bigwindow_map, kf_all, total_sec
    )
    item_kf_windows_map[item_id] = windows_result

    T_event = meta['T_event']

    # ── 突變點展開幀去重（kf_anchor強制保留，展開幀用DEDUP_THRESH_A）──

    all_kept_embs = []
    for r in windows_result:
        t_start     = round(r['window'][0], 2)
        new_sampled = []
        for t in r['sampled']:
            t_r   = round(t, 2)
            t_int = max(0, min(int(round(t)), len(embeddings) - 1))
            emb   = embeddings[t_int:t_int+1]
            if t_r == t_start:
                # kf_anchor 強制保留
                new_sampled.append(t)
                all_kept_embs.append(emb)
            elif all_kept_embs:
                vstack = np.vstack(all_kept_embs)
                # 【修正】NaN 防呆：跳過問題向量，保守保留該幀
                if np.any(np.isnan(emb)) or np.any(np.isnan(vstack)):
                    new_sampled.append(t)
                    all_kept_embs.append(emb)
                    continue
                sims = cosine_similarity(emb, vstack)[0]
                if sims.max() <= DEDUP_THRESH_A:
                    new_sampled.append(t)
                    all_kept_embs.append(emb)
            else:
                new_sampled.append(t)
                all_kept_embs.append(emb)
        r['sampled'] = new_sampled

    print(f'ID={item_id:4d}  T_event={T_event}s')
    print(f'  突變點序列：{[round(t,1) for t in kf_seq]}')
    for i, r in enumerate(windows_result):
        ws, we = r['window']
        print(f'  窗口{i+1}：{ws}s ~ {we}s（{we-ws:.1f}s），去重後={len(r["sampled"])}幀  {r["sampled"]}')

print(f'\n✅ 共處理 {len(item_kf_windows_map)} 題')


## Cell 6e：語意窗口取樣

T_event附近的密集取樣 + T_event前後的稀疏取樣，提供語意分析所需的幀序列。

**兩組固定取樣（不分問題類型）：**

| 組別 | 範圍 | 間距 | 幀數 |
|---|---|---|---|
| 組一（附近）| T_event前後各SEMANTIC_NEAR_N幀 | SEMANTIC_NEAR_INTERVAL=0.25s | 最多2N+1幀 |
| 組二（稀疏）| T_event往前/往後各SEMANTIC_FAR_N幀 | SEMANTIC_FAR_INTERVAL=2.0s | 最多2×FAR_N幀 |

組一和組二合併去重後作為C語意窗口的選幀結果。

**邊界處理：** 超出影片範圍的幀自動捨棄

**Optuna可優化：** `SEMANTIC_NEAR_N`、`SEMANTIC_NEAR_INTERVAL`、`SEMANTIC_FAR_N`、`SEMANTIC_FAR_INTERVAL`

**輸入：** `item_tevent_map`、`total_sec`

**輸出：** `item_semantic_map`（dict，key=item_id，value=選幀時間列表）


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 6e：語意窗口取樣（費波那契間距）
# ═══════════════════════════════════════════════════════════

def compute_semantic_times(T_event, total_sec,
                            near_n=SEMANTIC_NEAR_N,
                            near_interval=SEMANTIC_NEAR_INTERVAL,
                            fib_offsets=FIB_OFFSETS):
    """
    組一：T_event前後各near_n幀，間距near_interval秒
    組二：T_event往前/往後各取費波那契間距秒數
    合併去重，裁剪至[0, total_sec]
    """
    times = set()
    for i in range(-near_n, near_n + 1):
        t = round(T_event + i * near_interval, 2)
        if 0 <= t <= total_sec:
            times.add(t)
    for d in fib_offsets:
        for t in [round(T_event - d, 2), round(T_event + d, 2)]:
            if 0 <= t <= total_sec:
                times.add(t)
    return sorted(times)

item_semantic_map = {}
for item_id, meta in item_tevent_map.items():
    T_event = meta['T_event']
    times   = compute_semantic_times(T_event, total_sec)
    item_semantic_map[item_id] = times
    near_cutoff = SEMANTIC_NEAR_N * SEMANTIC_NEAR_INTERVAL + 0.01
    far_times = [t for t in times if abs(t - T_event) > near_cutoff]
    print(f'ID={item_id:4d}  T_event={T_event}s  共{len(times)}幀（費波那契：{len(far_times)}幀）')

print(f'\n✅ 共處理 {len(item_semantic_map)} 題')


## Cell 6f：即時窗口取樣

以 realtime 為錨點，捕捉觸發提問的視覺事件。

**兩組固定取樣：**

| 組別 | 範圍 | 間距 | 幀數 |
|---|---|---|---|
| 組一（附近）| realtime前後各REALTIME_NEAR_N幀 | REALTIME_NEAR_INTERVAL=0.25s | 最多2N+1幀 |
| 組二（往前）| realtime往前REALTIME_FAR_N幀 | REALTIME_FAR_INTERVAL=2.0s | 最多FAR_N幀 |

**設計原則：** realtime是精確錨點，往後不取（答案在提問之前）

**邊界處理：** 超出影片範圍的幀自動捨棄

**Optuna可優化：** `REALTIME_NEAR_N`、`REALTIME_NEAR_INTERVAL`、`REALTIME_FAR_N`、`REALTIME_FAR_INTERVAL`

**輸入：** `item_tevent_map`（含realtime）、`total_sec`

**輸出：** `item_realtime_map`（dict，key=item_id，value=選幀時間列表）


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 6f：即時窗口取樣（費波那契間距）
# ═══════════════════════════════════════════════════════════

def compute_realtime_times(realtime, total_sec,
                            near_n=REALTIME_NEAR_N,
                            near_interval=REALTIME_NEAR_INTERVAL,
                            fib_offsets=FIB_OFFSETS):
    """
    組一：realtime前後各near_n幀，間距near_interval秒
    組二：realtime往前費波那契間距秒數
    合併去重，裁剪至[0, total_sec]
    """
    times = set()
    for i in range(-near_n, near_n + 1):
        t = round(realtime + i * near_interval, 2)
        if 0 <= t <= total_sec:
            times.add(t)
    for d in fib_offsets:
        t = round(realtime - d, 2)
        if 0 <= t <= total_sec:
            times.add(t)
    return sorted(times)

item_realtime_map = {}
# 🌟 新增防呆機制：如果 all_items 不存在，就自動從 all_qa 抓取當前的題目
if 'all_items' not in globals():
    all_items = [i for i in all_qa if i['id'] == ITEM_ID] if 'ITEM_ID' in globals() else all_qa

for item in all_items:  # ← 現在這裡絕對找得到東西了
    item_id  = item['id']
    realtime = float(item['realtime'])
    times    = compute_realtime_times(realtime, total_sec)
    item_realtime_map[item_id] = times
    near_cutoff = REALTIME_NEAR_N * REALTIME_NEAR_INTERVAL + 0.01
    far_times = [t for t in times if abs(t - realtime) > near_cutoff]
    print(f'ID={item_id:4d}  realtime={realtime}s  共{len(times)}幀（費波那契：{len(far_times)}幀）')

print(f'\n✅ 共處理 {len(item_realtime_map)} 題')


## Cell 6g：統一時間戳 + 來源標籤整合

將所有窗口（D大窗口、A突變點、C語意、B即時）的選幀結果整合成統一結構。

**來源標籤：**
- `D 大窗口` — 大窗口逆CDF高斯取樣
- `A 突變點窗口N` — 第N個突變點窗口展開幀
- `C 語意窗口（附近）` — T_event附近密集幀
- `C 語意窗口（稀疏）` — T_event前後稀疏幀
- `B 即時窗口（附近）` — realtime附近密集幀
- `B 即時窗口（往前）` — realtime往前稀疏幀

**特別標籤（KEY）：**
- `KEY: T_event` — T_event本身
- `KEY: realtime` — realtime本身
- `KEY: kf_anchor` — 突變點本身（窗口起點，非展開幀）

**輸出：** `item_frame_map`（dict，key=item_id，value=sorted list of {time, sources}）


In [ ]:
# ==========================================
# Cell 6g：統一時間戳 + 來源標籤整合
# ==========================================
from collections import defaultdict

item_frame_map = {}

# 🌟 新增防呆機制
if 'all_items' not in globals():
    all_items = [i for i in all_qa if i['id'] == ITEM_ID] if 'ITEM_ID' in globals() else all_qa

for item in all_items:
    item_id  = item['id']
    realtime = float(item['realtime'])
    meta     = item_tevent_map[item_id]
    T_event  = meta['T_event']

    # 每個時間點可能有多個來源標籤
    time_labels = defaultdict(list)

    # ── D 大窗口 ──
    for t in item_bigwindow_map[item_id]['big_times']:
        time_labels[t].append('D 大窗口')

    # ── A 突變點窗口 ──
    kf_windows = item_kf_windows_map[item_id]
    for i, r in enumerate(kf_windows):
        t_start = r['window'][0]
        for t in r['sampled']:
            time_labels[t].append(f'A 突變點窗口{i+1}')
        # 窗口起點（突變點本身）特別標註
        time_labels[t_start].append('KEY: kf_anchor')

    # ── C 語意窗口 ──
    near_cutoff = SEMANTIC_NEAR_N * SEMANTIC_NEAR_INTERVAL + 0.01
    for t in item_semantic_map[item_id]:
        if abs(t - T_event) <= near_cutoff:
            time_labels[t].append('C 語意窗口（附近）')
        else:
            time_labels[t].append('C 語意窗口（稀疏）')

    # ── B 即時窗口 ──
    rt_near_cutoff = REALTIME_NEAR_N * REALTIME_NEAR_INTERVAL + 0.01
    for t in item_realtime_map[item_id]:
        if abs(t - realtime) <= rt_near_cutoff:
            time_labels[t].append('B 即時窗口（附近）')
        else:
            time_labels[t].append('B 即時窗口（往前）')

    # ── 特別標註 KEY ──
    t_event_r = round(T_event, 2)
    realtime_r = round(realtime, 2)
    time_labels[t_event_r].append('KEY: T_event')
    time_labels[realtime_r].append('KEY: realtime')

    # ── 整合成列表，按時間排序 ──
    frame_list = sorted([
        {'time': t, 'sources': list(set(labels))}
        for t, labels in time_labels.items()
    ], key=lambda x: x['time'])

    item_frame_map[item_id] = frame_list

    # 印出統計
    total = len(frame_list)
    key_frames = [f for f in frame_list if any('KEY' in s for s in f['sources'])]
    print(f'ID={item_id:4d}  T_event={T_event}s  realtime={realtime}s')
    print(f'  總幀數：{total}  KEY幀：{len(key_frames)}')
    print(f'  KEY幀時間：{[round(f["time"],2) for f in key_frames]}')
    print(f'  KEY幀標籤：{[f["sources"] for f in key_frames]}')

print(f'\n✅ 共處理 {len(item_frame_map)} 題')


## Cell 7a：大窗口幀輸出

針對大窗口選出的幀（已在Cell 6c完成去重）：
- **T_event**：原圖輸出（不resize）
- **其他幀**：resize至 `BIG_OUTPUT_SIZE`（160×90）

不做任何語意處理，YOLO/ViT留給語意窗口使用。

**token估計：**
- T_event（原圖 1280×720）≈ 1600 token
- 其他幀（160×90）≈ 20 token/幀

**輸入：** `item_bigwindow_map`（big_times，已去重）、`item_tevent_map`（T_event）

**輸出：** 大窗口幀圖片，儲存至 `/content/test_7a/`


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 7a：大窗口幀輸出
# ═══════════════════════════════════════════════════════════
import cv2, os, time

os.makedirs('/content/test_7a', exist_ok=True)

ITEM_ID_TEST = ITEM_ID

big_times = item_bigwindow_map[ITEM_ID_TEST]['big_times']
T_event   = item_tevent_map[ITEM_ID_TEST]['T_event']

print(f'大窗口幀數：{len(big_times)}')
print(f'T_event={T_event}s → 原圖')
print(f'其他幀 → {BIG_OUTPUT_SIZE}')

cap = cv2.VideoCapture(RAW_PATH)
results = []
t_start = time.time()

for t in big_times:
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
    ret, frame = cap.read()
    if not ret:
        print(f'  t={t}s 截圖失敗，跳過')
        continue

    t0 = time.time()
    out_path = f'/content/test_7a/t{t:.1f}s.jpg'

    if t == T_event:
        h, w = frame.shape[:2]
        scale = 420 / max(w, h)
        img = cv2.resize(frame, (int(w*scale), int(h*scale)))
        cv2.imwrite(out_path, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        tag = 'T_event（原圖）'
    else:
        h, w = frame.shape[:2]
        scale = 140 / max(w, h)
        img = cv2.resize(frame, (int(w*scale), int(h*scale)))
        cv2.imwrite(out_path, img, [cv2.IMWRITE_JPEG_QUALITY, 85])
        tag = f'縮圖{BIG_OUTPUT_SIZE}'

    elapsed = time.time() - t0
    results.append({'time': t, 'tag': tag, 'elapsed': elapsed})
    print(f'  t={t:6.1f}s  {tag}  耗時={elapsed:.3f}s')

cap.release()

# ── D-Window 拼圖（排除 T_event）──
from PIL import Image, ImageDraw, ImageFont

d_frames = [(r['time'], f'/content/test_7a/t{r["time"]:.1f}s.jpg')
            for r in results if r['tag'] != 'T_event（原圖）']

if d_frames:
    imgs = [Image.open(p) for _, p in d_frames]
    W, H = imgs[0].size  # 每格寬高（長邊240等比例後一致）
    strip = Image.new('RGB', (W, H * len(imgs)), (30, 30, 30))
    draw = ImageDraw.Draw(strip)
    for idx, ((t, _), img) in enumerate(zip(d_frames, imgs)):
        strip.paste(img, (0, idx * H))
        draw.rectangle([0, idx*H, W-1, idx*H + 18], fill=(0,0,0,180))
        draw.text((4, idx*H + 2), f'{t:.1f}s', fill=(255,255,0))
    strip_path = f'/content/test_7a/d_window_strip.jpg'
    strip.save(strip_path, quality=85)
    print(f'✅ D-Window 拼圖：{len(imgs)} 幀 → {strip_path}')




total_elapsed = time.time() - t_start

n_tevent = 1
n_other  = len(results) - n_tevent
print(f'\n✅ 共處理 {len(results)} 幀')
print(f'總耗時：{total_elapsed:.2f}s  平均：{total_elapsed/len(results):.3f}s/幀')
print(f'\nT_event（原圖）：1張 ≈ 1600 token')
print(f'其他幀（{BIG_OUTPUT_SIZE}）：{n_other}張 ≈ {n_other * 20} token')
print(f'大窗口總token估計：{1600 + n_other * 20} token')


## Cell 7b：突變點窗口幀輸出

針對突變點窗口選出的幀（已在Cell 6d完成去重）：
- **突變點本身（kf_anchor）**：resize至 `KF_ANCHOR_SIZE`（640×360，1/4原圖）
- **窗口內展開幀**：resize至 `KF_OUTPUT_SIZE`（224×126）

不做任何語意處理（YOLO/ViT留給語意窗口）。

**token估計：**
- kf_anchor（640×360）≈ 400 token/張
- 展開幀（224×126）≈ 40 token/幀

**輸入：** `item_kf_windows_map`（已去重）、`item_frame_map`（判斷KEY: kf_anchor）

**輸出：** 突變點窗口幀圖片，儲存至 `/content/test_7b/`


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 7b：突變點窗口幀輸出
# ═══════════════════════════════════════════════════════════
import cv2, os, time

os.makedirs('/content/test_7b', exist_ok=True)

ITEM_ID_TEST = ITEM_ID

kf_windows  = item_kf_windows_map[ITEM_ID_TEST]
frame_list  = item_frame_map[ITEM_ID_TEST]

kf_anchor_times = set(
    round(f['time'], 2)
    for f in frame_list
    if 'KEY: kf_anchor' in f['sources']
)

all_kf_times = sorted(set(
    round(t, 2)
    for r in kf_windows
    for t in r['sampled']
))

print(f'突變點窗口共{len(kf_windows)}個窗口，{len(all_kf_times)}幀')
print(f'kf_anchor時間點：{sorted(kf_anchor_times)}')
print(f'kf_anchor → {KF_ANCHOR_SIZE}  展開幀 → {KF_OUTPUT_SIZE}')

cap = cv2.VideoCapture(RAW_PATH)
results = []
t_start = time.time()

for t in all_kf_times:
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
    ret, frame = cap.read()
    if not ret:
        print(f'  t={t}s 截圖失敗，跳過')
        continue

    t0 = time.time()
    out_path = f'/content/test_7b/t{t:.2f}s.jpg'

    if t in kf_anchor_times:
        h, w = frame.shape[:2]
        scale = 420 / max(w, h)
        img = cv2.resize(frame, (int(w*scale), int(h*scale)))
        cv2.imwrite(out_path, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        tag = f'kf_anchor {KF_ANCHOR_SIZE}'
    else:
        h, w = frame.shape[:2]
        scale = 140 / max(w, h)
        img = cv2.resize(frame, (int(w*scale), int(h*scale)))
        cv2.imwrite(out_path, img, [cv2.IMWRITE_JPEG_QUALITY, 85])
        tag = f'縮圖{KF_OUTPUT_SIZE}'

    elapsed = time.time() - t0
    results.append({'time': t, 'tag': tag, 'elapsed': elapsed})
    print(f'  t={t:6.2f}s  {tag}  耗時={elapsed:.3f}s')

cap.release()


# ── A-Window 拼圖（按窗口分組）──
# ── A-Window 拼圖（全部展開幀合一，排除 kf_anchor）──
from PIL import Image, ImageDraw

kf_windows = item_kf_windows_map[ITEM_ID_TEST]
kf_anchor_set = set(round(r['window'][0], 2) for r in kf_windows)

all_expand = []
for r in kf_windows:
    for t in r['sampled']:
        t_r = round(t, 2)
        if t_r not in kf_anchor_set:
            path = f'/content/test_7b/t{t_r:.2f}s.jpg'
            if os.path.exists(path):
                all_expand.append((t_r, path))

all_expand.sort(key=lambda x: x[0])

if all_expand:
    imgs = [Image.open(p) for _, p in all_expand]
    W, H = imgs[0].size
    strip = Image.new('RGB', (W, H * len(imgs)), (30, 30, 30))
    draw = ImageDraw.Draw(strip)
    for idx, ((t, _), img) in enumerate(zip(all_expand, imgs)):
        strip.paste(img, (0, idx * H))
        draw.rectangle([0, idx*H, W-1, idx*H + 18], fill=(0,0,0))
        draw.text((4, idx*H + 2), f'{t:.2f}s', fill=(255,255,0))
    strip_path = '/content/test_7b/a_window_strip.jpg'
    strip.save(strip_path, quality=85)
    print(f'✅ A-Window 拼圖：{len(imgs)} 幀 → {strip_path}')






total_elapsed = time.time() - t_start

n_anchor = len(kf_anchor_times)
n_expand = len(results) - n_anchor
print(f'\n✅ 共處理 {len(results)} 幀')
print(f'總耗時：{total_elapsed:.2f}s  平均：{total_elapsed/len(results):.3f}s/幀')
print(f'\nkf_anchor（{KF_ANCHOR_SIZE}）：{n_anchor}張 ≈ {n_anchor * 400} token')
print(f'展開幀（{KF_OUTPUT_SIZE}）：{n_expand}張 ≈ {n_expand * 40} token')
print(f'突變點窗口總token估計：{n_anchor * 400 + n_expand * 40} token')


## Cell 7c：核心幀目標物標注（YOLO-World）+ 組一位移分析

**Part 1：7張核心幀標注**
- **T_event、realtime**：維持原圖（1280×720）做YOLO標注
- **kf_anchor突變點（5張）**：resize至640×360後做YOLO標注

**Part 2：組一位移分析**
- T_event附近4幀、realtime附近4幀：resize至640×360後YOLO標注
- 計算目標物位移趨勢和面積變化，輸出推測性文字描述

**閾值參數（Optuna可優化）：** `STU_MOVE_THRESH=0.02`、`STU_AREA_THRESH=0.10`

**輸出：** 標注圖至 `/content/test_7c/`，位移描述至 `motion_description_{ID}.txt`


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 7c：核心幀目標物標注（YOLO-World）+ 組一位移分析
# ═══════════════════════════════════════════════════════════
import cv2, os, time
import numpy as np

os.makedirs('/content/test_7c', exist_ok=True)

ITEM_ID_TEST = ITEM_ID

meta     = item_tevent_map[ITEM_ID_TEST]
T_event  = meta['T_event']
nouns    = meta['nouns']
item_data = next(i for i in all_items if i['id'] == ITEM_ID_TEST)
realtime  = float(item_data['realtime'])

print(f'T_event={T_event}s  realtime={realtime}s')
print(f'查詢名詞：{nouns}')

# ── YOLO-World設定查詢詞 ──
if nouns:
    yolo_world.to('cpu')
    yolo_world.set_classes(nouns)
    yolo_world.to(device)
    print(f'YOLO-World查詢詞：{nouns}')

# ── 標注函數 ──
def detect_and_annotate(frame, nouns, conf_thresh=0.25):
    img = frame.copy()
    detected = []
    if not nouns:
        return img, detected
    results = yolo_world(frame, verbose=False)
    h, w = frame.shape[:2]
    for r in results:
        if r.boxes is None:
            continue
        for box in r.boxes:
            conf = float(box.conf[0])
            if conf < conf_thresh:
                continue
            cls_id = int(box.cls[0])
            cls_name = nouns[cls_id] if cls_id < len(nouns) else str(cls_id)
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cx = (x1 + x2) / 2 / w
            cy = (y1 + y2) / 2 / h
            area = (x2 - x1) * (y2 - y1) / (w * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            label = f'{cls_name} {conf:.2f}'
            cv2.putText(img, label, (x1, y1 - 8),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            detected.append({'class': cls_name, 'conf': conf,
                            'cx': cx, 'cy': cy, 'area': area,
                            'box': [x1, y1, x2, y2]})
    return img, detected

# ── 位移分析函數 ──
def analyze_motion(detections_seq, nouns):
    descriptions = []
    for noun in nouns:
        positions = []
        areas = []
        for dets in detections_seq:
            matches = [d for d in dets if d['class'] == noun]
            if matches:
                best = max(matches, key=lambda x: x['conf'])
                positions.append((best['cx'], best['cy']))
                areas.append(best['area'])
            else:
                positions.append(None)
                areas.append(None)

        valid_pos = [(i, p) for i, p in enumerate(positions) if p is not None]

        if len(valid_pos) < 2:
            descriptions.append(f'{noun}：僅在部分幀中偵測到，位置趨勢難以判斷')
            continue

        dx_list = []
        dy_list = []
        for i in range(len(valid_pos) - 1):
            _, p1 = valid_pos[i]
            _, p2 = valid_pos[i + 1]
            dx_list.append(p2[0] - p1[0])
            dy_list.append(p2[1] - p1[1])

        avg_dx = np.mean(dx_list)
        avg_dy = np.mean(dy_list)

        valid_areas = [a for a in areas if a is not None]
        area_change = (valid_areas[-1] - valid_areas[0]) / valid_areas[0] if len(valid_areas) >= 2 else 0

        h_desc = ''
        v_desc = ''
        d_desc = ''

        if abs(avg_dx) > STU_MOVE_THRESH:
            dir_h = '右' if avg_dx > 0 else '左'
            h_desc = f'似乎傾向往{dir_h}移動'
        if abs(avg_dy) > STU_MOVE_THRESH:
            dir_v = '下' if avg_dy > 0 else '上'
            v_desc = f'似乎傾向往{dir_v}移動'
        if abs(area_change) > STU_AREA_THRESH:
            d_desc = '可能正在靠近' if area_change > 0 else '可能正在遠離'

        parts = [p for p in [h_desc, v_desc, d_desc] if p]
        if parts:
            desc = f'{noun}：' + '，'.join(parts)
        else:
            desc = f'{noun}：位置與大小大致穩定'
        descriptions.append(desc)

    return descriptions

# ── 主流程 ──
cap = cv2.VideoCapture(RAW_PATH)
results_log = {}
t_all = time.time()

# ── Part 1：核心幀標注 ──
print('\n── Part 1：7張核心幀 ──')
frame_list = item_frame_map[ITEM_ID_TEST]
kf_anchor_times = sorted(set(
    round(f['time'], 2)
    for f in frame_list
    if 'KEY: kf_anchor' in f['sources'] and round(f['time'], 2) != round(T_event, 2)
))

core_frames = [
    (T_event, 'T_event'),
    (realtime, 'realtime')
]

for t, tag in core_frames:
    t_read = min(t, total_sec - 0.5)
    cap.set(cv2.CAP_PROP_POS_MSEC, t_read * 1000)
    ret, frame = cap.read()
    if not ret:
        print(f'  ⚠️ 讀取失敗：{tag} t={t}s')
        continue
    if tag == 'realtime':
        frame_yolo = frame
        _, detected = detect_and_annotate(frame_yolo, nouns)
        out_path = f'/content/test_7c/{tag}_t{t:.1f}s.jpg'
        cv2.imwrite(out_path, frame)
    else:
        h, w = frame.shape[:2]
        scale = 420 / max(w, h)
        frame_yolo = cv2.resize(frame, (int(w*scale), int(h*scale)))
        _, detected = detect_and_annotate(frame_yolo, nouns)
        # 不存圖，只取偵測數值
    results_log[tag] = detected
    print(f'  {tag} t={t:.1f}s  偵測到{len(detected)}個目標')
    for d in detected:
        print(f'    → {d["class"]} conf={d["conf"]:.2f}')

# ── Part 2：T_event附近4幀 + realtime附近4幀位移分析 ──
print('\n── Part 2：組一位移分析 ──')

for anchor_t, anchor_tag in [(T_event, 'T_event'), (realtime, 'realtime')]:
    near_times = [
        round(anchor_t - 0.5, 2),
        round(anchor_t - 0.25, 2),
        round(anchor_t + 0.25, 2),
        round(anchor_t + 0.5, 2),
    ]
    near_times = [t for t in near_times if 0 <= t <= total_sec]

    seq_detections = []
    for t in near_times:
        t_read = min(t, total_sec - 0.5)
        cap.set(cv2.CAP_PROP_POS_MSEC, t_read * 1000)
        ret, frame = cap.read()
        if not ret:
            seq_detections.append([])
            continue
        h, w = frame.shape[:2]
        scale = 420 / max(w, h)
        frame_small = cv2.resize(frame, (int(w*scale), int(h*scale)))
        _, detected = detect_and_annotate(frame_small, nouns)
        out_path = f'/content/test_7c/{anchor_tag}_near_t{t:.2f}s.jpg'
        cv2.imwrite(out_path, frame_small)
        seq_detections.append(detected)

    if nouns and any(seq_detections):
        motion_desc = analyze_motion(seq_detections, nouns)
        print(f'  {anchor_tag}附近位移分析：')
        for desc in motion_desc:
            print(f'    {desc}')
        results_log[f'{anchor_tag}_motion'] = motion_desc
    else:
        print(f'  {anchor_tag}附近：無偵測結果')

cap.release()
total_elapsed = time.time() - t_all
print(f'\n✅ 完成，總耗時：{total_elapsed:.2f}s')
print(f'\n位移分析結果摘要：')
for key in ['T_event_motion', 'realtime_motion']:
    if key in results_log:
        print(f'  {key}：')
        for desc in results_log[key]:
            print(f'    {desc}')

motion_tevent_path = f'/content/test_7c/motion_description_{ITEM_ID_TEST}_tevent.txt'
with open(motion_tevent_path, 'w') as f_out:
    f_out.write(f'Item ID: {ITEM_ID_TEST}\n')
    f_out.write(f'Question: {question}\n\n')
    f_out.write('=== Motion Analysis near T_event ===\n\n')
    if 'T_event_motion' in results_log:
        for desc in results_log['T_event_motion']:
            f_out.write(f'  {desc}\n')
print(f'✅ Motion Analysis (T_event) 儲存至：{motion_tevent_path}')

motion_realtime_path = f'/content/test_7c/motion_description_{ITEM_ID_TEST}_realtime.txt'
with open(motion_realtime_path, 'w') as f_out:
    f_out.write(f'Item ID: {ITEM_ID_TEST}\n')
    f_out.write(f'Question: {question}\n\n')
    f_out.write('=== Motion Analysis near realtime ===\n\n')
    if 'realtime_motion' in results_log:
        for desc in results_log['realtime_motion']:
            f_out.write(f'  {desc}\n')
print(f'✅ Motion Analysis (realtime) 儲存至：{motion_realtime_path}')

## Cell 7d：動作序列事件定位（線三、線六、線七）

針對ASI/FPD問題，用三條線定位T_event+1（下一事件關鍵幀）：

**線三：CLIP-Seg目標區域相似度**
```
seg_mask = CLIP-Seg(frame_t, question_noun)
sim3(t) = cosine(v_seg_Tevent, v_seg_t)
```
sim3下降至低點 → 進入下一情境

**線六：選項名詞vs問題名詞差值**
```
diff6_i(t) = cosine(v_seg_t, v_question_noun) - cosine(v_seg_t, v_option_noun_i)
```
線三和線六共用同一次CLIP-Seg

**線七：動作語意對齊（用已有全片embeddings）**
```
delta_t = embeddings[t+1] - embeddings[t]
diff7_i(t) = cosine(delta_t, v_verb_question) - cosine(delta_t, v_verb_option_i)
```

**方向性：**
- after → 從T_event往後找
- before → 從T_event往前找
- 無after/before → 兩方向都找
- realtime → 從realtime往回找

**最終判斷：**
每個選項的 score = drop6 + drop7，score最大的選項為答案

**輸出：** 三條線曲線圖 + T_event+1截圖


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 7d：動作序列事件定位（線三、線六、線七）
# ═══════════════════════════════════════════════════════════
import cv2, os, time, torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs('/content/test_7d', exist_ok=True)

ITEM_ID_TEST = ITEM_ID

meta      = item_tevent_map[ITEM_ID_TEST]
T_event   = meta['T_event']
nouns     = meta['nouns']
verbs     = meta['verbs']
is_after  = meta['is_after']
is_before = meta['is_before']

item_data = next(i for i in all_items if i['id'] == ITEM_ID_TEST)
realtime  = float(item_data['realtime'])
options   = item_data['options']
question  = item_data['question']

print(f'Q: {question}')
print(f'T_event={T_event}s  realtime={realtime}s')
print(f'nouns={nouns}  verbs={verbs}')
print(f'after={is_after}  before={is_before}')
print(f'options={options}')

# ── 名詞篩選（物件名詞，排除人物詞）──
PERSON_WORDS = {'person', 'people', 'man', 'woman', 'i', 'me', 'he', 'she', 'they'}
obj_nouns = [n for n in nouns if n.lower() not in PERSON_WORDS]
if not obj_nouns:
    obj_nouns = nouns  # 若過濾後為空，保留原始名詞
print(f'物件名詞（線三/六用）：{obj_nouns}')


def safe_cos(a, b):
    if np.any(np.isnan(a)) or np.any(np.isnan(b)):
        return np.zeros((a.shape[0], b.shape[0]))
    if np.all(a == 0) or np.all(b == 0):
        return np.zeros((a.shape[0], b.shape[0]))
    return cosine_similarity(a, b)


# ── 選項名詞抽取 ──
def extract_option_nouns(option_text):
    doc = nlp(option_text)
    nouns_opt = [t.text for t in doc if t.pos_ in ('NOUN', 'PROPN')]
    nouns_obj  = [n for n in nouns_opt if n.lower() not in PERSON_WORDS]
    return nouns_obj if nouns_obj else nouns_opt

option_nouns = [extract_option_nouns(opt) for opt in options]
option_verbs = []
for opt in options:
    doc = nlp(opt)
    option_verbs.append([t.text for t in doc if t.pos_ == 'VERB'])

print(f'選項名詞：{option_nouns}')
print(f'選項動詞：{option_verbs}')

# ── CLIP文字向量 ──
def encode_text(words):
    if not words:
        return None
    text = ' '.join(words)
    inputs = siglip_processor(text=[text], return_tensors='pt', padding=True).to(device)
    with torch.no_grad():
        emb = siglip_model.text_model(**inputs).pooler_output.detach().cpu().numpy()
    return emb / (np.linalg.norm(emb) + 1e-8)

v_q_noun = encode_text(obj_nouns)
v_q_verb = encode_text(verbs)
v_opt_nouns = [encode_text(ns) for ns in option_nouns]
v_opt_verbs = [encode_text(vs) for vs in option_verbs]

# ── CLIP-Seg函數 ──
def clipseg_region_vector(img_pil, text_query):
    inputs = clipseg_processor(
        text=[text_query],
        images=[img_pil],
        return_tensors='pt',
        padding=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = clipseg_model(**inputs)
    mask = torch.sigmoid(outputs.logits[0]).cpu().numpy()
    mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
    img_np = np.array(img_pil.resize((224, 224)))
    mask_resized = cv2.resize(mask, (224, 224))
    masked = img_np * mask_resized[:, :, np.newaxis]
    masked_pil = Image.fromarray(masked.astype(np.uint8))
    inputs = siglip_processor(images=masked_pil, return_tensors='pt').to(device)
    with torch.no_grad():
        vec = siglip_model.vision_model(**inputs).pooler_output.detach().cpu().numpy()
    vec = vec / (np.linalg.norm(vec) + 1e-8)
    return vec

# ── 決定搜索方向 ──
search_ranges = []
if is_after:
    times_after = [round(T_event + d, 1) for d in FIB_OFFSETS
                   if T_event + d <= total_sec]
    search_ranges.append(('after_Tevent', times_after))
elif is_before:
    times_before = [round(T_event - d, 1) for d in FIB_OFFSETS
                    if T_event - d >= 0]
    search_ranges.append(('before_Tevent', times_before))
else:
    times_after  = [round(T_event + d, 1) for d in FIB_OFFSETS
                    if T_event + d <= total_sec]
    times_before = [round(T_event - d, 1) for d in FIB_OFFSETS
                    if T_event - d >= 0]
    search_ranges.append(('after_Tevent', times_after))
    search_ranges.append(('before_Tevent', times_before))

# ── realtime軸：費波那契往前取偵 ──
times_rt = [round(realtime - d, 1) for d in FIB_OFFSETS
            if realtime - d >= 0]

# ── T_event的CLIP-Seg向量（線三的anchor）──
print('\n計算T_event的CLIP-Seg向量...')
cap = cv2.VideoCapture(RAW_PATH)
cap.set(cv2.CAP_PROP_POS_MSEC, T_event * 1000)
ret, frame = cap.read()
img_pil_Tevent = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
query_text = ' '.join(obj_nouns)
v_seg_Tevent = clipseg_region_vector(img_pil_Tevent, query_text)
print(f'✅ T_event CLIP-Seg向量計算完成')

# ── 主計算迴圈（T_event軸：線三+線六+線七）──
all_results = {}

for range_name, times in search_ranges:
    if not times:
        continue
    print(f'\n── {range_name}：{len(times)}幀 ──')

    sim3_vals  = []
    diff6_vals = [[] for _ in options]
    diff7_vals = [[] for _ in options]
    valid_times = []

    for t in times:
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
        ret, frame = cap.read()
        if not ret:
            continue

        img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        # 線三 + 線六：共用CLIP-Seg
        v_seg_t = clipseg_region_vector(img_pil, query_text)

        # 線三 ── 【修正】safe_cos
        s3 = float(safe_cos(v_seg_Tevent, v_seg_t)[0][0])
        sim3_vals.append(s3)

        # 線六（每個選項）── 【修正】safe_cos
        if v_q_noun is not None:
            for i, v_opt in enumerate(v_opt_nouns):
                if v_opt is not None:
                    sq = float(safe_cos(v_seg_t, v_q_noun)[0][0])
                    so = float(safe_cos(v_seg_t, v_opt)[0][0])
                    diff6_vals[i].append(sq - so)
                else:
                    diff6_vals[i].append(0)

        # 線七：用已有全片embeddings ── 【修正】safe_cos
        t_int = int(round(t))
        t_int = max(0, min(t_int, len(embeddings) - 2))
        delta = embeddings[t_int + 1] - embeddings[t_int]
        delta = delta / (np.linalg.norm(delta) + 1e-8)

        if v_q_verb is not None:
            for i, v_opt_v in enumerate(v_opt_verbs):
                if v_opt_v is not None:
                    sq = float(safe_cos(delta.reshape(1,-1), v_q_verb)[0][0])
                    so = float(safe_cos(delta.reshape(1,-1), v_opt_v)[0][0])
                    diff7_vals[i].append(sq - so)
                else:
                    diff7_vals[i].append(0)

        valid_times.append(t)
        print(f'  t={t:.1f}s  sim3={s3:.3f}', end='')
        if diff6_vals[0]:
            print(f'  diff6={[f"{d[-1]:.3f}" for d in diff6_vals]}', end='')
        print()

    all_results[range_name] = {
        'times':      valid_times,
        'sim3':       sim3_vals,
        'diff6':      diff6_vals,
        'diff7':      diff7_vals,
    }

# ── realtime軸計算（線三+線七）──
print(f'\n── before_realtime（提問脈絡）：{len(times_rt)}幀 ──')

cap.set(cv2.CAP_PROP_POS_MSEC, realtime * 1000)
ret, frame_rt = cap.read()
if ret:
    img_pil_rt = Image.fromarray(cv2.cvtColor(frame_rt, cv2.COLOR_BGR2RGB))
    v_seg_realtime = clipseg_region_vector(img_pil_rt, query_text)
else:
    v_seg_realtime = v_seg_Tevent  # fallback

sim3_rt_vals  = []
sim7_rt_vals  = []
valid_times_rt = []

for t in times_rt:
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
    ret, frame = cap.read()
    if not ret:
        continue

    img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    v_seg_t = clipseg_region_vector(img_pil, query_text)

    # 線三（realtime為anchor）── 【修正】safe_cos
    s3 = float(safe_cos(v_seg_realtime, v_seg_t)[0][0])
    sim3_rt_vals.append(s3)

    # 線七 ── 【修正】safe_cos
    t_int = int(round(t))
    t_int = max(0, min(t_int, len(embeddings) - 2))
    delta = embeddings[t_int + 1] - embeddings[t_int]
    delta = delta / (np.linalg.norm(delta) + 1e-8)
    if v_q_verb is not None:
        s7 = float(safe_cos(delta.reshape(1,-1), v_q_verb)[0][0])
    else:
        s7 = 0.0
    sim7_rt_vals.append(s7)

    valid_times_rt.append(t)
    print(f'  t={t:.1f}s  sim3_rt={s3:.3f}  sim7_rt={s7:.3f}')

rt_results = {
    'times': valid_times_rt,
    'sim3':  sim3_rt_vals,
    'sim7':  sim7_rt_vals,
}

cap.release()


# ── 選項得分（只從T_event軸計算）──
print('\n── 選項得分（T_event軸）──')
OPTION_LETTERS = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
noun_scores_total = [0.0] * len(options)
verb_scores_total = [0.0] * len(options)

for range_name, res in all_results.items():
    for i in range(len(options)):
        drop6 = (max(res['diff6'][i]) - min(res['diff6'][i])) if res['diff6'][i] else 0
        drop7 = (max(res['diff7'][i]) - min(res['diff7'][i])) if res['diff7'][i] else 0
        noun_scores_total[i] += drop6
        verb_scores_total[i] += drop7
        print(f'  {range_name} 選項{i+1}（{options[i][:25]}）：drop6={drop6:.3f} drop7={drop7:.3f}')

total_scores = [noun_scores_total[i] + verb_scores_total[i] for i in range(len(options))]
best_noun  = int(np.argmax(noun_scores_total))
best_verb  = int(np.argmax(verb_scores_total))
best_total = int(np.argmax(total_scores))

print('\n── 總分排名 ──')
for i, opt in enumerate(options):
    print(f'  選項{OPTION_LETTERS[i]}（{opt[:25]}）：noun={noun_scores_total[i]:.3f} verb={verb_scores_total[i]:.3f} total={total_scores[i]:.3f}')
print(f'  → 名詞最高：{OPTION_LETTERS[best_noun]}  動詞最高：{OPTION_LETTERS[best_verb]}  總分最高：{OPTION_LETTERS[best_total]}')

scores_text = 'Next Event Analysis Scores (T_event axis):\n'
for i, opt in enumerate(options):
    scores_text += (f'  Option {OPTION_LETTERS[i]} ({opt[:30]}): '
                   f'noun={noun_scores_total[i]:.3f} '
                   f'verb={verb_scores_total[i]:.3f} '
                   f'total={total_scores[i]:.3f}\n')
llm_hints = [scores_text]

print('\n── LLM建議文字描述（針對下一個事件）──')
for h in llm_hints:
    print(f'  {h}')

item_7d_hints = {ITEM_ID_TEST: llm_hints}

hints_tevent_path = f'/content/test_7d/llm_hints_{ITEM_ID_TEST}_tevent.txt'
with open(hints_tevent_path, 'w') as f_out:
    f_out.write(f'Item ID: {ITEM_ID_TEST}\n')
    f_out.write(f'Question: {question}\n\n')
    f_out.write('=== Next Event Analysis Scores (T_event axis) ===\n\n')
    f_out.write(llm_hints[0])
print(f'✅ System Hints (T_event) 儲存至：{hints_tevent_path}')

hints_realtime_path = f'/content/test_7d/llm_hints_{ITEM_ID_TEST}_realtime.txt'
with open(hints_realtime_path, 'w') as f_out:
    f_out.write(f'Item ID: {ITEM_ID_TEST}\n')
    f_out.write(f'Question: {question}\n\n')
    f_out.write('=== System Analysis (Question Context - from realtime axis) ===\n\n')
    if rt_results and rt_results['times']:
        sim3_trend = 'increasing' if rt_results['sim3'][-1] > rt_results['sim3'][0] else 'decreasing'
        sim7_trend = 'increasing' if rt_results['sim7'][-1] > rt_results['sim7'][0] else 'decreasing'
        f_out.write(f'Object similarity trend near realtime (sim3): {sim3_trend}\n')
        f_out.write(f'Action verb similarity trend near realtime (sim7): {sim7_trend}\n')
        f_out.write(f'sim3 range: {min(rt_results["sim3"]):.3f} ~ {max(rt_results["sim3"]):.3f}\n')
        f_out.write(f'sim7 range: {min(rt_results["sim7"]):.3f} ~ {max(rt_results["sim7"]):.3f}\n')
print(f'✅ System Hints (realtime) 儲存至：{hints_realtime_path}')

# ── YOLO標注函數 ──
def yolo_annotate(frame, nouns):
    annotated = frame.copy()
    if not nouns:
        return annotated
    yolo_results = yolo_world(frame, verbose=False)
    for r in yolo_results:
        if r.boxes is None:
            continue
        for box in r.boxes:
            conf = float(box.conf[0])
            if conf < 0.25:
                continue
            cls_id   = int(box.cls[0])
            cls_name = nouns[cls_id] if cls_id < len(nouns) else str(cls_id)
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(annotated, f'{cls_name} {conf:.2f}',
                       (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return annotated

cap2 = cv2.VideoCapture(RAW_PATH)

if 'after_Tevent' in all_results:
    res_next = all_results['after_Tevent']
elif all_results:
    res_next = list(all_results.values())[0]
else:
    res_next = None

if res_next and res_next['times']:
    min_idx = int(np.argmin(res_next['sim3']))
    t_next  = res_next['times'][min_idx]
    cap2.set(cv2.CAP_PROP_POS_MSEC, t_next * 1000)
    ret, frame = cap2.read()
    if ret:
        h, w = frame.shape[:2]
        scale = 420 / max(w, h)
        frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
        out_path = f'/content/test_7d/T_event_plus1_t{t_next:.1f}s.jpg'
        cv2.imwrite(out_path, frame)
        print(f'\n✅ T_event+1截圖（YOLO標注）：t={t_next}s → {out_path}')

# ── T_event_minus1：before_Tevent 的 sim3 最低點 ──
if 'before_Tevent' in all_results and all_results['before_Tevent']['times']:
    res_prev = all_results['before_Tevent']
    min_idx_prev = int(np.argmin(res_prev['sim3']))
    t_prev = res_prev['times'][min_idx_prev]
    cap2.set(cv2.CAP_PROP_POS_MSEC, t_prev * 1000)
    ret, frame = cap2.read()
    if ret:
        h, w = frame.shape[:2]
        scale = 420 / max(w, h)
        frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
        out_path = f'/content/test_7d/T_event_minus1_t{t_prev:.1f}s.jpg'
        cv2.imwrite(out_path, frame)
        print(f'✅ T_event-1截圖：t={t_prev}s → {out_path}')

t_rt_minus1 = max(0, realtime - 1)
cap2.set(cv2.CAP_PROP_POS_MSEC, t_rt_minus1 * 1000)
ret, frame = cap2.read()
if ret:
    out_path = f'/content/test_7d/realtime_minus1_t{t_rt_minus1:.1f}s.jpg'
    cv2.imwrite(out_path, frame)
    print(f'✅ realtime-1截圖（YOLO標注）：t={t_rt_minus1}s → {out_path}')

# ── CLIP 相似度比較（T_event vs realtime）──
clip_sim_result = {}
tevent_path   = f'/content/test_7a/t{round(T_event, 1):.1f}s.jpg'
realtime_path = f'/content/test_7c/realtime_t{realtime:.1f}s.jpg'

if nouns and os.path.exists(tevent_path) and os.path.exists(realtime_path):
    try:
        noun_text = ' '.join(nouns[:3])
        inputs = siglip_processor(text=[noun_text], return_tensors='pt', padding=True).to(device)
        with torch.no_grad():
            txt_emb = siglip_model.text_model(**inputs).pooler_output.detach().cpu().numpy()
        txt_emb = txt_emb / np.linalg.norm(txt_emb)

        from PIL import Image as _Image
        def clip_img_sim(img_path):
            img_pil = _Image.open(img_path).convert('RGB')
            inputs = siglip_processor(images=img_pil, return_tensors='pt').to(device)
            with torch.no_grad():
                img_emb = siglip_model.vision_model(**inputs).pooler_output.detach().cpu().numpy()
            img_emb = img_emb / np.linalg.norm(img_emb)
            return float(cosine_similarity(txt_emb, img_emb)[0][0])

        clip_sim_result['T_event']  = clip_img_sim(tevent_path)
        clip_sim_result['realtime'] = clip_img_sim(realtime_path)
        clip_sim_result['noun_text'] = noun_text
        print(f'📊 CLIP 相似度 → T_event: {clip_sim_result["T_event"]:.3f}  realtime: {clip_sim_result["realtime"]:.3f}')
    except Exception as e:
        print(f'⚠️ CLIP 相似度計算失敗：{e}')

# ── YOLO-World 關鍵物件截圖（五張原圖，每張最多2個物件）──
os.makedirs('/content/test_7d/crops', exist_ok=True)

# 五張原圖的來源定義
crop_sources = []

# T_event（從 cap2 讀）
cap2.set(cv2.CAP_PROP_POS_MSEC, T_event * 1000)
ret, frame = cap2.read()
if ret:
    crop_sources.append(('T_event', T_event, frame.copy()))

# T_event_plus1
if res_next and res_next['times']:
    cap2.set(cv2.CAP_PROP_POS_MSEC, t_next * 1000)
    ret, frame = cap2.read()
    if ret:
        crop_sources.append(('T_event_plus1', t_next, frame.copy()))

# T_event_minus1
if 'before_Tevent' in all_results and all_results['before_Tevent']['times']:
    cap2.set(cv2.CAP_PROP_POS_MSEC, t_prev * 1000)
    ret, frame = cap2.read()
    if ret:
        crop_sources.append(('T_event_minus1', t_prev, frame.copy()))

# realtime
t_realtime_read = min(realtime, total_sec - 0.5)
cap2.set(cv2.CAP_PROP_POS_MSEC, t_realtime_read * 1000)
ret, frame = cap2.read()
if ret:
    crop_sources.append(('realtime', realtime, frame.copy()))

# realtime_minus1
cap2.set(cv2.CAP_PROP_POS_MSEC, t_rt_minus1 * 1000)
ret, frame = cap2.read()
if ret:
    crop_sources.append(('realtime_minus1', t_rt_minus1, frame.copy()))

# YOLO-World 偵測 + 截圖
print('\n── YOLO-World 關鍵物件截圖 ──')
for src_name, src_t, frame in crop_sources:
    h, w = frame.shape[:2]
    results_yw = yolo_world(frame, verbose=False)
    detections = []
    for r in results_yw:
        if r.boxes is None:
            continue
        for box in r.boxes:
            conf = float(box.conf[0])
            if conf < 0.25:
                continue
            cls_id = int(box.cls[0])
            cls_name = nouns[cls_id] if cls_id < len(nouns) else str(cls_id)
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            detections.append((conf, cls_name, x1, y1, x2, y2))

    # 按信心值排序，取前兩個
    detections.sort(key=lambda x: x[0], reverse=True)
    for conf, cls_name, x1, y1, x2, y2 in detections[:2]:
        # 加 10% padding
        pad_x = int((x2 - x1) * 0.1)
        pad_y = int((y2 - y1) * 0.1)
        x1c = max(0, x1 - pad_x)
        y1c = max(0, y1 - pad_y)
        x2c = min(w, x2 + pad_x)
        y2c = min(h, y2 + pad_y)
        crop = frame[y1c:y2c, x1c:x2c]
        if crop.size == 0:
            continue
        ch, cw = crop.shape[:2]
        crop_scale = 280 / max(cw, ch)
        if crop_scale < 1.0:
            crop = cv2.resize(crop, (int(cw*crop_scale), int(ch*crop_scale)))
        fname = f'{src_name}_t{src_t:.1f}s_{cls_name}.jpg'
        out_path = f'/content/test_7d/crops/{fname}'
        cv2.imwrite(out_path, crop)
        print(f'  ✅ {fname}  conf={conf:.2f}')

    if not detections:
        print(f'  ⚠️ {src_name} t={src_t:.1f}s：無偵測結果')

cap2.release()

## Cell 7e：材料清單整理

彙整所有要送給LLM的圖片路徑和標籤，輸出`item_dedup_map`供Cell 8使用。

**來源：**
- 大窗口縮圖（Cell 7a，去重已在Cell 6c完成）
- 突變點展開幀縮圖（Cell 7b，去重已在Cell 6d完成）
- 核心幀YOLO標注圖（Cell 7c）
- T_event+1、realtime-1截圖（Cell 7d）
- 位移描述txt（Cell 7c）
- LLM建議txt（Cell 7d）

**輸出：** `item_dedup_map`（dict，key=item_id，value=材料清單）


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 7e：材料清單整理
# ═══════════════════════════════════════════════════════════
import os

ITEM_ID_TEST = ITEM_ID

item_data = next(i for i in all_items if i['id'] == ITEM_ID_TEST)
T_event   = item_tevent_map[ITEM_ID_TEST]['T_event']
realtime  = float(item_data['realtime'])

materials = {
    'images': [],   # {'path': ..., 'label': ...}
    'texts':  [],   # {'path': ..., 'label': ...}
}

# ── 分類1：大窗口（D）──
big_times = item_bigwindow_map[ITEM_ID_TEST]['big_times']

# T_event 原圖單獨收
t_ev = round(T_event, 1)
t_ev_path = f'/content/test_7a/t{t_ev:.1f}s.jpg'
if os.path.exists(t_ev_path):
    materials['images'].append({'path': t_ev_path, 'label': f'[T_event @ {t_ev}s] Key moment'})

# D-Window 收拼圖
strip_path = '/content/test_7a/d_window_strip.jpg'
if os.path.exists(strip_path):
    materials['images'].append({'path': strip_path, 'label': '[D-Window] Scene overview strip (chronological, timestamps shown)'})

# ── 分類2：突變點展開幀（A）+ kf_anchor YOLO標注（來自test_7c）──
kf_windows = item_kf_windows_map[ITEM_ID_TEST]
strip_path = '/content/test_7b/a_window_strip.jpg'
if os.path.exists(strip_path):
    materials['images'].append({'path': strip_path, 'label': '[A-Window] Narrative segment strip (chronological, timestamps shown)'})


# kf_anchor 從 test_7b 取原圖（top-3 by CLIP sim，排除 strip）
kf_anchor_candidates = []
if os.path.exists('/content/test_7b'):
    for fname in sorted(os.listdir('/content/test_7b')):
        if not fname.endswith('.jpg') or 'strip' in fname:
            continue
        t_str = fname.replace('.jpg', '').split('t')[-1].replace('s', '')
        t_val = float(t_str)
        t_int = max(0, min(int(round(t_val)), len(embeddings) - 1))
        emb   = embeddings[t_int]
        if v_q_noun is not None:
            sim = float(np.dot(emb, v_q_noun.flatten()))
        else:
            sim = -t_val
        kf_anchor_candidates.append((sim, t_val, f'/content/test_7b/{fname}'))

kf_anchor_candidates.sort(key=lambda x: x[0], reverse=True)
for sim, t_val, path in kf_anchor_candidates[:3]:
    label = f'[CORE: kf_anchor] Scene transition t={t_val:.1f}s'
    materials['images'].append({'path': path, 'label': label})




if os.path.exists('/content/test_7d'):
    for fname in sorted(os.listdir('/content/test_7d')):
        if not fname.endswith('.jpg'):
            continue
        if 'T_event_plus1' in fname:
            t_val = fname.replace('.jpg','').split('t')[-1].replace('s','')
            label = f'[NEXT EVENT @ {t_val}s] Frame after T_event'
            materials['images'].append({'path': f'/content/test_7d/{fname}', 'label': label})
        elif 'T_event_minus1' in fname:
            t_val = fname.replace('.jpg','').split('t')[-1].replace('s','')
            label = f'[PREV EVENT @ {t_val}s] Frame before T_event'
            materials['images'].append({'path': f'/content/test_7d/{fname}', 'label': label})

# ── 分類4：realtime即時分析 ──
if os.path.exists('/content/test_7c'):
    for fname in sorted(os.listdir('/content/test_7c')):
        if not fname.endswith('.jpg'):
            continue
        if 'realtime' in fname and 'near' not in fname and 'minus' not in fname:
            path  = f'/content/test_7c/{fname}'
            label = '[CORE: realtime] Object detection at question moment'
            materials['images'].append({'path': path, 'label': label})

if os.path.exists('/content/test_7d'):
    for fname in sorted(os.listdir('/content/test_7d')):
        if not fname.endswith('.jpg'):
            continue
        if 'realtime_minus1' in fname:
            t_val = fname.replace('.jpg','').split('t')[-1].replace('s','')
            label = f'[PRE-REALTIME @ {t_val}s] Frame before question moment'
            materials['images'].append({'path': f'/content/test_7d/{fname}', 'label': label})

# ── crops：YOLO 關鍵物件截圖 ──
crops_dir = '/content/test_7d/crops'
if os.path.exists(crops_dir):
    for fname in sorted(os.listdir(crops_dir)):
        if not fname.endswith('.jpg'):
            continue
        path = f'{crops_dir}/{fname}'
        if fname.startswith('T_event_t'):
            label = f'[T_event CROP] {fname}'
            cat = 1
        elif fname.startswith('T_event_plus1') or fname.startswith('T_event_minus1'):
            label = f'[EVENT CROP] {fname}'
            cat = 3
        elif fname.startswith('realtime'):
            label = f'[REALTIME CROP] {fname}'
            cat = 4
        else:
            continue
        materials['images'].append({'path': path, 'label': label, 'cat': cat})




# ── 文字材料（分類3和分類4分開）──
for path, label in [
    (f'/content/test_7c/motion_description_{ITEM_ID_TEST}_tevent.txt',  'Motion Analysis T_event'),
    (f'/content/test_7d/llm_hints_{ITEM_ID_TEST}_tevent.txt',           'System Hints T_event'),
    (f'/content/test_7c/motion_description_{ITEM_ID_TEST}_realtime.txt','Motion Analysis realtime'),
    (f'/content/test_7d/llm_hints_{ITEM_ID_TEST}_realtime.txt',         'System Hints realtime'),
]:
    if os.path.exists(path):
        materials['texts'].append({'path': path, 'label': label})

# CLIP 相似度結果存進 materials
materials['clip_sim'] = clip_sim_result



item_dedup_map = {ITEM_ID_TEST: materials}

print(f'✅ 材料清單整理完成')
print(f'  圖片：{len(materials["images"])}張')
for img in materials['images']:
    print(f'    {img["label"]}')
print(f'  文字：{len(materials["texts"])}份')
for txt in materials['texts']:
    print(f'    {txt["label"]}')

## Cell 8：打包材料 + Prompt設計 + Claude API答題

從 `item_dedup_map`（Cell 7e）取得所有材料，依四分類組成structured prompt送給Claude API。

**四分類資料：**
1. 大窗口（`[D-Window]`、`[T_event]`）：場景概覽，提供時間軸背景
2. 突變點（`[A-Window*]`、`[CORE: kf_anchor YOLO]`）：敘事段落邊界與內容
3. T_event語意（`[CORE: T_event YOLO]`、`[NEXT EVENT]`、System Hints）：關鍵時刻物件分析
4. realtime即時（`[CORE: realtime YOLO]`、`[PRE-REALTIME]`、Motion Analysis）：提問當下脈絡

**答題格式：** 只回答A/B/C/D，不符格式最多重問3次

**貢獻值：** 答題後請LLM對四分類各給0~1的貢獻分數


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 8：打包材料 + CoT Prompt設計 + Claude API答題 (優化版)
# ═══════════════════════════════════════════════════════════
import anthropic, base64, os, re, time
from pathlib import Path

# ╔══════════════════════════════╗
# ║        資料整理              ║
# ╚══════════════════════════════╝

# ── 基本設定 ──
ITEM_ID_TEST = ITEM_ID
MAX_RETRIES  = 3
OPTION_LETTERS = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
UTA_CONFIDENCE_THRESHOLD = 3

# ── 題目資訊讀取 ──
item_data = next(i for i in all_items if i['id'] == ITEM_ID_TEST)
question  = item_data['question']
options   = item_data['options']
gt        = item_data['gt']
realtime  = float(item_data['realtime'])
T_event   = item_tevent_map[ITEM_ID_TEST]['T_event']

print(f'Q: {question}')
print(f'T_event={T_event}s  realtime={realtime}s')
print(f'正確答案：{OPTION_LETTERS[gt]}（{options[gt]}）')
options_str = '\n'.join([f'{OPTION_LETTERS[i]}. {opt}' for i, opt in enumerate(options)])

# ── UTA 偵測 ──
UTA_KEYWORDS = ['unable to answer', 'cannot be determined', 'not visible',
                'cannot determine', 'not enough information', 'unclear']

def is_uta_option(opt_text):
    return any(kw in opt_text.lower() for kw in UTA_KEYWORDS)

uta_idx = None
for i, opt in enumerate(options):
    if is_uta_option(opt):
        uta_idx = i
        break

if uta_idx is not None:
    print(f'🔍 偵測到 Unable to Answer 選項：{OPTION_LETTERS[uta_idx]}（{options[uta_idx]}）')
else:
    print(f'ℹ️ 本題無 Unable to Answer 選項，停用刪去法')

# ── 材料讀取 + 文字解析 ──
materials = item_dedup_map[ITEM_ID_TEST]
images    = materials['images']
texts     = materials['texts']

def read_txt(path):
    with open(path) as f:
        return f.read()

motion_tevent_text   = ''
motion_realtime_text = ''
hints_tevent_text    = ''
hints_realtime_text  = ''
for t in texts:
    if t['label'] == 'Motion Analysis T_event':
        motion_tevent_text   = read_txt(t['path'])
    elif t['label'] == 'Motion Analysis realtime':
        motion_realtime_text = read_txt(t['path'])
    elif t['label'] == 'System Hints T_event':
        hints_tevent_text    = read_txt(t['path'])
    elif t['label'] == 'System Hints realtime':
        hints_realtime_text  = read_txt(t['path'])

# ── 圖片分類 ──
cat1, cat2, cat3, cat4 = [], [], [], []
for img in images:
    lbl = img['label']
    explicit_cat = img.get('cat')
    if explicit_cat == 1 or '[D-Window' in lbl or '[T_event' in lbl:
        cat1.append(img)
    elif explicit_cat == 2 or '[A-Window' in lbl or '[CORE: kf_anchor' in lbl or 'kf_anchor' in lbl:
        cat2.append(img)
    elif explicit_cat == 3 or '[NEXT EVENT' in lbl or '[PREV EVENT' in lbl:
        cat3.append(img)
    elif explicit_cat == 4 or '[CORE: realtime' in lbl or '[PRE-REALTIME' in lbl:
        cat4.append(img)

print(f'\n分類1（大窗口）：{len(cat1)}張')
print(f'分類2（突變點）：{len(cat2)}張')
print(f'分類3（T_event語意）：{len(cat3)}張')
print(f'分類4（realtime即時）：{len(cat4)}張')

# ── Prompt 定義 ──
system_prompt = """You are a video understanding analyst. You are given video frames and analytical data to answer a multiple-choice question about what happens in the video at `[realtime]`.

**Your materials:**
- Category 1: [T_event] key moment frame + [D-Window] chronological background strip (chronological, timestamps shown) + cropped object images at T_event.
- Category 2: [A-Window] chronological narrative strip (timestamps shown) + [kf_anchor] scene transitions.
- Category 3: [PREV EVENT] frame before T_event + [NEXT EVENT] frame after T_event + cropped object images near T_event + T_event axis analysis (motion trends, option scores).
- Category 4: [realtime] + [PRE-REALTIME] frames + cropped object images near realtime + realtime axis analysis (motion trends).

**Task:** You CANNOT see beyond `[realtime]`. Follow the focus instructions below to deduce the answer.

Output your reasoning inside <thought_process> tags before answering:
Step 1 — Identify in 1 sentence: subject, object, action, and time direction (current/past/future/sequential).
Step 2 — What is the strongest visual evidence from the frames? (2-3 sentences max)
Step 3 — State your answer and reasoning.
"""

cat_labels = {
    1: 'Category 1: Scene Overview — T_event key moment + D-Window background strip (chronological) + object crops at T_event',
    2: 'Category 2: Narrative — A-Window strip (chronological) + kf_anchor scene transitions',
    3: 'Category 3: Event Context — PREV/NEXT EVENT frames + object crops near T_event + motion & option score analysis',
    4: 'Category 4: Realtime Context — realtime + PRE-REALTIME frames + object crops near realtime + motion analysis',
}

cat_texts = {
    3: (motion_tevent_text + '\n' + hints_tevent_text).strip(),
    4: (motion_realtime_text + '\n' + hints_realtime_text).strip(),
}

def img_to_b64(path):
    with open(path, 'rb') as f:
        return base64.standard_b64encode(f.read()).decode('utf-8')

def make_image_block(img):
    b64 = img_to_b64(img['path'])
    ext = Path(img['path']).suffix.lower()
    mt  = 'image/jpeg' if ext in ['.jpg', '.jpeg'] else 'image/png'
    return [
        {'type': 'text', 'text': f'--- {img["label"]} ---'},
        {'type': 'image', 'source': {'type': 'base64', 'media_type': mt, 'data': b64}}
    ]

def build_user_content(include_contribution_request=True):
    content = []
    ordered_cats = {1: cat1, 2: cat2, 3: cat3, 4: cat4}
    q_type  = item_tevent_map[ITEM_ID_TEST]['q_type']
    q_lower = question.lower()

    if q_type == 'action':
        strict_order = [1, 2, 3, 4]
        system_focus = "\nCRITICAL FOCUS: Pay attention to the action sequence and physical inertia of the subject."
        if any(p in q_lower for p in ['what is he doing', 'what is she doing', 'what is the person doing', 'what does']):
            system_focus += "\n🚨 TIME ANCHOR: Current action. Prioritize CAT4 frames."
        if any(p in q_lower for p in ['what did', 'what happened', 'what was']):
            system_focus += "\n🚨 PAST EVENT: Past action. Focus on CAT2 and CAT3 history."
        if any(p in q_lower for p in ['after', 'next', 'then', 'following']):
            system_focus += "\n🚨 SEQUENCE: Question asks what happens AFTER an event. Trace the action sequence in CAT3 and CAT4."
        if any(p in q_lower for p in ['before', 'prior', 'previously']):
            system_focus += "\n🚨 PRIOR EVENT: Question asks what happened BEFORE. Focus on CAT2 and CAT3 earlier frames."
    elif q_type == 'spatial':
        strict_order = [2, 3, 4, 1]
        system_focus = "\nCRITICAL FOCUS: Pay close attention to spatial positions and environmental context."
        if any(p in q_lower for p in ['where', 'which side', 'which hand', 'which direction']):
            system_focus += "\n🚨 TRACKING: Locate the object's LAST known position and observe its final background environment."
        if any(p in q_lower for p in ['direction', 'facing', 'towards', 'away']) or any(w in q_lower for w in ['move', 'moving', 'go', 'going', 'attack', 'walk', 'run']):
            system_focus += "\n🚨 DIRECTION: Compare multiple frames chronologically. Do NOT rely on a single static frame."
    elif q_type == 'attribute':
        strict_order = [1, 2, 4, 3]
        system_focus = "\nCRITICAL FOCUS: Pay close attention to visual attributes (color, quantity, shape, text). Prioritize Category 3 and Category 4 close-up frames."
        if any(p in q_lower for p in ['how many', 'count', 'number of']):
            system_focus += "\n🚨 COUNTING: Track carefully. Do NOT count the same object multiple times across frames."
        if any(p in q_lower for p in ['what color', 'what colour', 'what kind', 'what type', 'which', 'what size']):
            system_focus += "\n🚨 ATTRIBUTE: Focus on CAT3 and CAT4 close-up crops for fine-grained visual detail."
    else:
        strict_order = [1, 2, 3, 4]
        system_focus = ""

    print(f"🧠 Prompt 組裝邏輯：啟動 {q_type.upper()} 題型注意力引導 (傳送順序 {strict_order})")

    clip_sim = materials.get('clip_sim', {})
    if clip_sim:
        sim_tevent   = clip_sim.get('T_event', 0)
        sim_realtime = clip_sim.get('realtime', 0)
        noun_text    = clip_sim.get('noun_text', '')
        print(f'  📊 CLIP 相似度 → T_event({T_event:.1f}s): {sim_tevent:.3f}  realtime({realtime:.1f}s): {sim_realtime:.3f}')
        priority = f'T_event @ {T_event:.1f}s' if sim_tevent >= sim_realtime else f'realtime @ {realtime:.1f}s'
        system_focus += f"\n🎯 KEY FRAME: Prioritize [{priority}] for '{noun_text}'."

    for cat_num in strict_order:
        imgs = ordered_cats[cat_num]
        if not imgs:
            continue
        content.append({'type': 'text', 'text': f'\n## {cat_labels[cat_num]}'})
        for img in imgs:
            content.extend(make_image_block(img))
        if cat_num in cat_texts and cat_texts[cat_num].strip():
            content.append({'type': 'text', 'text': f'\n**Objective Motion Data:**\n{cat_texts[cat_num]}'})

    all_letters = [OPTION_LETTERS[i] for i in range(len(options))]
    if uta_idx is not None:
        uta_letter       = OPTION_LETTERS[uta_idx]
        concrete_letters = [OPTION_LETTERS[i] for i in range(len(options)) if i != uta_idx]
        confidence_instruction = f"""
<confidence>
{chr(10).join([f"{l}: [0-5]" for l in concrete_letters])}
</confidence>
Rate each option 0-5. If ALL below {UTA_CONFIDENCE_THRESHOLD}, select {uta_letter}.
"""
    else:
        confidence_instruction = f"""
<confidence>
{chr(10).join([f"{l}: [0-5]" for l in all_letters])}
</confidence>
Rate each option 0-5. Select the highest.
"""

    contrib_format = """
CONTRIBUTION (0.0-1.0): Category 1: X.X, Category 2: X.X, Category 3: X.X, Category 4: X.X""" if include_contribution_request else ''

    content.append({'type': 'text', 'text': f"""
## QUESTION
{question}

## OPTIONS
{options_str}

## INSTRUCTIONS
Review all materials above.{system_focus}{confidence_instruction}
- Respond inside <thought_process> tags, then <answer>X</answer>.{contrib_format}
"""})
    return content


# ╔══════════════════════════════╗
# ║        送給 API              ║
# ╚══════════════════════════════╝

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
answer  = None
retries = 0
total_input_tokens  = 0
total_output_tokens = 0
t_start = time.time()

while retries < MAX_RETRIES and answer is None:
    content = build_user_content(include_contribution_request=True)

    if retries > 0:
        content.append({'type': 'text', 'text': f'Your previous response format was invalid. Please ensure you output your final choice EXACTLY inside <answer> tags. (attempt {retries+1}/{MAX_RETRIES})'})

    response = client.messages.create(
        model='claude-sonnet-4-5-20250929',
        max_tokens=2000,
        system=system_prompt,
        messages=[{'role': 'user', 'content': content}]
    )
    total_input_tokens  += response.usage.input_tokens
    total_output_tokens += response.usage.output_tokens
    raw = response.content[0].text.strip()

    # ── 信心分數解析 + UTA 刪去法 ──
    confidence_triggered = False
    conf_scores = {}
    conf_match = re.search(r'<confidence>(.*?)</confidence>', raw, re.DOTALL | re.IGNORECASE)
    if conf_match:
        for line in conf_match.group(1).strip().split('\n'):
            m = re.match(r'\s*([A-H])\s*[:：]\s*(\d+)', line.strip())
            if m:
                conf_scores[m.group(1).upper()] = int(m.group(2))
        print(f'\n── 信心分數 ──')
        for k, v in conf_scores.items():
            print(f'  {k}: {v}')

    if uta_idx is not None and conf_scores:
        uta_letter = OPTION_LETTERS[uta_idx]
        concrete_scores = {k: v for k, v in conf_scores.items() if k != uta_letter}
        if concrete_scores and all(v < UTA_CONFIDENCE_THRESHOLD for v in concrete_scores.values()):
            answer = uta_letter
            confidence_triggered = True
            print(f'🚨 刪去法觸發：所有具體選項信心分數均低於 {UTA_CONFIDENCE_THRESHOLD}，強制選擇 {uta_letter}（{options[uta_idx]}）')

    # ── answer 解析 ──
    if not confidence_triggered:
        match = re.search(r'<answer>\s*([A-H])\s*</answer>', raw, re.IGNORECASE)
        if not match:
            match = re.search(r'<answer>.*?([A-H]).*?</answer>', raw, re.IGNORECASE | re.DOTALL)
        if match:
            answer = match.group(1).upper()
        else:
            retries += 1
            print(f'⚠️ 無效回答格式（第{retries}次）：未找到 <answer> 標籤。部分輸出內容：{raw[-100:]}')

elapsed = time.time() - t_start


# ╔══════════════════════════════╗
# ║       分析與儲存             ║
# ╚══════════════════════════════╝

# ── 貢獻值解析 ──
contrib = {'cat1': None, 'cat2': None, 'cat3': None, 'cat4': None}
if answer:
    for cat_num, val in re.findall(r'Category (\d).*?:\s*([0-9.]+)', raw):
        key = f'cat{cat_num}'
        if key in contrib:
            contrib[key] = float(val)

# ── 結果輸出 ──
correct = (answer == OPTION_LETTERS[gt]) if answer else False
print(f'\n── 答題結果 ──')
print(f'解析答案：{answer}')
print(f'正確答案：{OPTION_LETTERS[gt]}（{options[gt]}）')
print(f'刪去法觸發：{"是" if confidence_triggered else "否"}')
print(f'結果：{"✅ 正確" if correct else "❌ 錯誤"}')
print(f'重試次數：{retries}')
print(f'耗時：{elapsed:.1f}s')
print(f'tokens：input={total_input_tokens} output={total_output_tokens}')
print(f'\n── 四分類貢獻值 ──')
for k, v in contrib.items():
    print(f'  {k}：{v}')

# ── item_result 儲存 ──
item_result = {
    ITEM_ID_TEST: {
        'answer':               answer,
        'correct':              correct,
        'gt':                   OPTION_LETTERS[gt],
        'retries':              retries,
        'confidence_triggered': confidence_triggered,
        'confidence_scores':    conf_scores,
        'contribution':         contrib,
        'tokens':               {'input': total_input_tokens, 'output': total_output_tokens}
    }
}
print(f'\n✅ 答題完成')

## Cell 9：計分 + 貢獻度記錄

每題答題後計算分類貢獻度，並累積到跨題統計。

**貢獻度加權：**
- 答對 → 貢獻值 × 1.0
- 答錯 → 貢獻值 × -0.5

**輸出：** `results/{youtube_id}.json`，結構參考OVO-Bench評測格式


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 9：計分 + 貢獻度記錄
# ═══════════════════════════════════════════════════════════
import json, os

os.makedirs('/content/results', exist_ok=True)

# ── 從Cell 8取得結果 ──
res                  = item_result[ITEM_ID_TEST]
answer               = res['answer']
correct              = res['correct']
contrib              = res['contribution']
tokens               = res['tokens']
confidence_triggered = res.get('confidence_triggered', False)  # 【新增】

# ── 從解答本查出這一題的加權分數 ──
item_id_str = str(ITEM_ID_TEST)
weighted_score = 0.0
if item_id_str in weight_table:
    weighted_score = weight_table[item_id_str].get(answer, 0.0)

# 貢獻度加權（答對×1，答錯×-0.5）
weight = 1.0 if correct else -0.5

CAT_NAMES = {
    'cat1': '1 大窗口',
    'cat2': '2 突變點窗口',
    'cat3': '3 T_event語意',
    'cat4': '4 realtime即時',
}

weighted_contrib = {}
for k, v in contrib.items():
    if v is not None:
        weighted_contrib[k] = round(v * weight, 4)
    else:
        weighted_contrib[k] = None

# ── 動態從題庫抓取這題的 TASK 名稱 ──
item_data = next((i for i in all_items if i['id'] == ITEM_ID_TEST), None)
TASK = item_data.get('task', 'OVO_TEST')

# ── 建立本題記錄 ──
item_record = {
    'item_id':             ITEM_ID_TEST,
    'task':                TASK,
    'q_type':              q_type,
    'question':            question,
    'gt':                  OPTION_LETTERS[gt],
    'llm_answer':          answer,
    'correct':             correct,
    'weighted_score':      weighted_score,
    'confidence_triggered': confidence_triggered,  # 【新增】
    'T_event':             float(T_event),
    'realtime':            float(realtime),
    'elapsed_sec': round(time.time() - t_item_start, 1) if 't_item_start' in globals() else None,
    'stage_times': stage_times if 'stage_times' in globals() else {},
    'confidence_scores': res.get('confidence_scores', {}),
    'tokens':              tokens,
    'contribution_raw':      contrib,
    'contribution_weighted': weighted_contrib,
}

print(f'── 本題記錄 ──')
print(f'  ID={ITEM_ID_TEST}  task={TASK}  correct={correct}  weight={weight}')
print(f'  刪去法觸發：{"是" if confidence_triggered else "否"}')
print(f'  原始貢獻值：{contrib}')
print(f'  加權貢獻值：{weighted_contrib}')

# ── 單題結果 ──
#item_result_data = {
#    'correct': int(correct),
#    'total':   1,
#    'accuracy': 1.0 if correct else 0.0,
#    'category_scores': {
#        CAT_NAMES[k]: {'value': v, 'weighted': weighted_contrib.get(k)}
#        for k, v in contrib.items()
#    },
#    'detail': [item_record]
#}

# ── 儲存JSON ──
out_path = f'/content/results/{TASK}_{ITEM_ID_TEST}_{EXP_NAME}.json'

result_data = {
    'task':                TASK,
    'item_id':             ITEM_ID_TEST,
    'exp_name':            EXP_NAME,
    'correct':             int(correct),
    'accuracy':            1.0 if correct else 0.0,
    'weighted_score':      weighted_score,
    'confidence_triggered': confidence_triggered,  # 【新增】
    'parameters': {
        "BIG_WINDOW_BASE_SIGMA":  BIG_WINDOW_BASE_SIGMA,
        "BIG_WINDOW_K":           BIG_WINDOW_K,
        "BIG_WINDOW_EPS":         BIG_WINDOW_EPS,
        "BIG_WINDOW_BASE_N":      BIG_WINDOW_BASE_N,
        "MIN_INTERVAL_BIG":       MIN_INTERVAL_BIG,
        "MAX_N_SMALL":            MAX_N_SMALL,
        "MIN_INTERVAL_SMALL":     MIN_INTERVAL_SMALL,
        "K_SIGMOID":              K_SIGMOID,
        "SEMANTIC_NEAR_N":        SEMANTIC_NEAR_N,
        "SEMANTIC_NEAR_INTERVAL": SEMANTIC_NEAR_INTERVAL,
        "REALTIME_NEAR_N":        REALTIME_NEAR_N,
        "REALTIME_NEAR_INTERVAL": REALTIME_NEAR_INTERVAL,
        "STU_MOVE_THRESH":        STU_MOVE_THRESH,
        "STU_AREA_THRESH":        STU_AREA_THRESH,
        "DEDUP_THRESH_D":         DEDUP_THRESH_D,
        "DEDUP_THRESH_A":         DEDUP_THRESH_A,
        "CLIP_DEDUP_THRESH":      CLIP_DEDUP_THRESH,
        "K":                      K,
        "ALPHA":                  ALPHA,
        "BETA":                   BETA,
    },
    'category_scores': {
        CAT_NAMES[k]: {'value': v, 'weighted': weighted_contrib.get(k)}
        for k, v in contrib.items()
    },
    'detail': [item_record]
}

with open(out_path, 'w') as f_out:
    json.dump(result_data, f_out, ensure_ascii=False, indent=2)

# ── 同步到 Google Drive ──
drive_results_dir = f'{DRIVE_ROOT}/ovo_results'
os.makedirs(drive_results_dir, exist_ok=True)
drive_path = f'{drive_results_dir}/{TASK}_{ITEM_ID_TEST}_{EXP_NAME}.json'
with open(drive_path, 'w') as f_out:
    json.dump(result_data, f_out, ensure_ascii=False, indent=2)

print(f'\n✅ 結果儲存至：{drive_path}')

## Cell 10：批次執行控制

設定要跑的題目 ID 列表，自動依序執行完整 pipeline。

**使用方式：**
1. 先跑 Cell 1~4（環境設定 + 模型載入，只需一次）
2. 修改 `run_ids` 放入想跑的題目 ID
3. 執行此 Cell，系統自動逐題跑完整 pipeline
4. 結果累積在 `session_results`，每題完成後自動儲存 JSON

**跑完後臨時截圖會自動清理，釋放空間**


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 10：批次執行控制
# ═══════════════════════════════════════════════════════════
import traceback, time, os, gc, torch, shutil, json

# ── 設定要跑的題目 ID 列表（自由設定）──
run_ids = [
    # PROBLEM


    1176, 1178, 1186, 1191, 1194, 1195, 1196, 1198, 1200, 1203,
    342, 343, 351, 353, 364, 380, 389, 397, 401, 417, 421, 431, 432, 433, 439, 442, 450, 451, 476, 480,

    1357, 1362, 1372, 1373, 1374, 1377, 1381, 1388, 1395, 1397, 1405, 1408, 1410, 1416, 1438, 1442, 1446, 1459, 1461, 1465,
    829, 840, 850, 851, 864, 865, 868, 871, 875, 910, 912, 933, 935, 938, 945, 959, 965, 979, 985, 990,
    658, 671, 674, 675, 676, 695, 716, 717, 731, 732, 740, 754, 755, 760, 764, 767, 775, 778, 781, 802,

]

# ── 執行單題的函數 ──
def run_one_item(item_id):
    """對單一題目執行完整 pipeline（Cell 5~9）"""
    # 🌟 修改 2a：加上 EXP_NAME
    global ITEM_ID, ITEM_ID_TEST, all_items, item, TASK, RAW_PATH, EXP_NAME
    global embeddings, timestamps, delta_embeddings, kf_all
    global total_sec, T_total, full_mu, full_sigma
    global item_tevent_map, item_bigwindow_map, item_kf_windows_map
    global item_semantic_map, item_realtime_map, item_frame_map
    global item_dedup_map, item_result, session_results

    # 設定題目
    ITEM_ID      = item_id
    ITEM_ID_TEST = item_id

    matched = [q for q in all_qa if q['id'] == ITEM_ID]
    if not matched:
        print(f'  ⚠️ 找不到 ITEM_ID={ITEM_ID}，跳過')
        return False

    item      = matched[0]
    TASK      = item.get('task', 'OVO_TEST')
    all_items = [item]

    # 🌟 修改 2b：同步我們剛剛在 Cell 5 修好的 RAW_PATH 組合邏輯
    video_id = str(item.get('youtube_id', item.get('video_name', item['id'])))
    if video_id in video_map:
        RAW_PATH = os.path.join(VIDEO_DIR, video_map[video_id])
    else:
        RAW_PATH = os.path.join(VIDEO_DIR, f"{video_id}.mp4")

    if not os.path.exists(RAW_PATH):
        print(f'  ⚠️ 影片不存在：{RAW_PATH}，跳過')
        return False

    print(f'  task={TASK}  Q: {item["question"][:60]}')

    try:
        global t_item_start
        t_item_start = time.time()
        stage_times = {}

        # Cell 5：CLIP編碼全片
        t0 = time.time()
        exec(compile(''.join(nb_cells[9]), '<cell9>', 'exec'), globals())
        stage_times['cell5'] = round(time.time() - t0, 1)

        # Cell 6b~6g：選幀
        t0 = time.time()
        for idx in [12, 14, 16, 18, 20, 22]:
            exec(compile(''.join(nb_cells[idx]), f'<cell{idx}>', 'exec'), globals())
        stage_times['cell6'] = round(time.time() - t0, 1)

        # Cell 7a~7e：截圖輸出
        t0 = time.time()
        for idx in [24, 26, 28, 30, 32]:
            exec(compile(''.join(nb_cells[idx]), f'<cell{idx}>', 'exec'), globals())
        stage_times['cell7'] = round(time.time() - t0, 1)

        # Cell 8：答題
        t0 = time.time()
        exec(compile(''.join(nb_cells[34]), '<cell34>', 'exec'), globals())
        stage_times['cell8'] = round(time.time() - t0, 1)

        # Cell 9：計分
        t0 = time.time()
        globals()['stage_times'] = stage_times  # 推進 globals 讓 Cell 9 能讀到
        exec(compile(''.join(nb_cells[36]), '<cell36>', 'exec'), globals())
        stage_times['cell9'] = round(time.time() - t0, 1)
        globals()['stage_times'] = stage_times  # 補上 cell9 的時間

        # Cell 9 跑完後補寫 stage_times 進 JSON
        out_path = f'/content/results/{TASK}_{ITEM_ID_TEST}_{EXP_NAME}.json'
        if os.path.exists(out_path):
            with open(out_path) as f:
                saved = json.load(f)
            if saved.get('detail'):
                saved['detail'][0]['stage_times'] = stage_times
            with open(out_path, 'w') as f:
                json.dump(saved, f, ensure_ascii=False, indent=2)

        return True

    except Exception as e:
        print(f'  ❌ 失敗：{e}')
        traceback.print_exc()
        return False

    finally:
        # 清理臨時截圖
        for d in ['/content/test_7a', '/content/test_7b',
                  '/content/test_7c', '/content/test_7d']:
            if os.path.exists(d):
                shutil.rmtree(d)
        # 強制清理GPU記憶體
        import matplotlib.pyplot as plt
        plt.close('all')
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


# ── 讀取當前notebook的cell內容 ──
import ipynbname
nb_path = '/content/ovo_stm_eval_v8_valid.ipynb'  # ← 你上傳到Colab的notebook路徑
with open(nb_path) as f:
    _nb = json.load(f)
nb_cells = [cell['source'] for cell in _nb['cells']]

# ── 初始化session_results ──
if 'session_results' not in globals():
    session_results = {
        'correct': 0, 'total': 0,
        'category_scores': {
            'cat1': {'cumulative': 0.0, 'count': 0},
            'cat2': {'cumulative': 0.0, 'count': 0},
            'cat3': {'cumulative': 0.0, 'count': 0},
            'cat4': {'cumulative': 0.0, 'count': 0},
        },
        'detail': []
    }

# ── 批次執行 ──
print(f'共 {len(run_ids)} 題待執行')
print('='*60)

batch_correct = 0
batch_total   = 0
batch_failed  = []
t_batch_start = time.time()

for i, item_id in enumerate(run_ids):
    print(f'\n[{i+1}/{len(run_ids)}] ID={item_id}')
    t0 = time.time()

    success = run_one_item(item_id)
    elapsed = time.time() - t0

    if success:
        batch_total += 1
        if item_result.get(item_id, {}).get('correct', False):
            batch_correct += 1
        acc = batch_correct / batch_total
        print(f'  ⏱ {elapsed:.1f}s  累積：{batch_correct}/{batch_total}（{acc:.1%}）')
    else:
        batch_failed.append(item_id)
        print(f'  ⏱ {elapsed:.1f}s  ❌ 失敗')

total_elapsed = time.time() - t_batch_start
print('\n' + '='*60)
print(f'✅ 批次完成')
print(f'成功：{batch_total}題  正確：{batch_correct}題  準確率：{batch_correct/max(batch_total,1):.1%}')
print(f'失敗：{len(batch_failed)}題  {batch_failed}')
print(f'總耗時：{total_elapsed/60:.1f}分鐘  平均：{total_elapsed/max(batch_total,1):.1f}s/題')
